# Notebook 88 — A5 Contextual Safe-Cleanup Model Training

**Runtime:** GPU. Select a T4 GPU, then **Runtime → Run all**.

## Purpose

Train a development-only contextual model which starts from the frozen **A1** page and may clean an A1-preserved ambiguous pixel only when observable local evidence supports **ANNOTATION** strongly enough.

The operational rule is deliberately asymmetric:

- **A5 CLEAN** → replace the A1 pixel with the frozen local paper estimate.
- **A5 KEEP / UNCERTAIN** → retain the exact A1 byte.
- Outside the authoritative reviewed removal mask → no change.

## Scientific boundary

Notebook 88 uses the completed Notebook 87 paired-HITL development run as labelled development evidence. Clean truth is used only through Notebook 87's post-lock labels. Clean image pixels and truth-derived arrays are prohibited inference channels.

The notebook:

1. resolves one completed Notebook 87 run;
2. verifies its review and prediction locks;
3. builds observable 33×33 tensors for A1-preserved ambiguous pixels;
4. trains compact CNNs under source-group out-of-fold separation;
5. fits and persists fold-specific isotonic calibration;
6. scores every eligible held-out pixel;
7. applies conservative CLEAN / KEEP / UNCERTAIN rules;
8. creates complete A1-versus-A5 pages;
9. reports annotation cleanup, genuine-print loss, catastrophic concentrations and page/source stability;
10. packages all training artefacts needed for a later freeze step.

**This is development training, not independent validation or production authorisation.**

# 0. Mount Drive and install dependencies

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib joblib

# 1. Configuration, constants and run identity

In [ ]:
# @title
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc, shutil, time

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N87_ROOT = ROOT / 'paired_hitl_development' / 'runs'
OUTPUT_ROOT = ROOT / 'a5_contextual_safe_cleanup_v88'

# Optional exact completed Notebook 87 run. Leave None to use the latest complete run.
SOURCE_N87_RUN_OVERRIDE = None

SEED = 42
PATCH = 33
HALF = PATCH // 2
BATCH = 512
EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_PER_PAGE_CLASS = 25_000
BOOTSTRAP_RESAMPLES = 10_000

# The model estimates p(annotation-safe-to-clean).
# These are predeclared development operating points, not production thresholds.
CLEAN_THRESHOLDS = [0.995, 0.990, 0.980, 0.950]
PRIMARY_CLEAN_THRESHOLD = 0.995
MIN_CLEAN_RUN = 2

# Labels inherited from Notebook 87 truth construction.
LABEL_PRINT = 0
LABEL_ANNOTATION = 1
LABEL_UNCERTAIN = 2

# Observable-only channels. The exact ordered list is saved and hashed.
CHANNELS = [
    "observed",
    "local_normalised",
    "reviewed_mask",
    "mask_depth",
    "ambiguous",
    "p_annotation",
    "p_print_value",
    "p_print_available",
    "edge",
    "horizontal_support",
    "vertical_support",
    "compact_support",
]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Notebook 88 requires a GPU runtime. Select a T4 GPU and rerun.')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_notebook88_a5_training')
RUN = OUTPUT_ROOT / RUN_ID
INPUTS = RUN / 'inputs'
TABLES = RUN / 'tables'
MODELS = RUN / 'models'
CALIBRATORS = RUN / 'calibrators'
PAGES = RUN / 'pages'
AUDIT = RUN / 'audit'
VISUALS = RUN / 'visuals'
UPLOAD = RUN / 'upload'

for folder in [RUN, INPUTS, TABLES, MODELS, CALIBRATORS, PAGES, AUDIT, VISUALS, UPLOAD]:
    folder.mkdir(parents=True, exist_ok=False)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def object_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str, separators=(',', ':')).encode()
    ).hexdigest()


def array_hash(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()

print('Device:', DEVICE, torch.cuda.get_device_name(0))
print('Output:', RUN)

# 2. Resolve and verify one completed Notebook 87 source run

In [ ]:
# @title
N87_REQUIRED = [
    'prediction_lock.json',
    'notebook87_verdict.json',
    'tables/paired_hitl_ambiguous_truth.parquet',
    'tables/truth_scores_by_page.csv',
    'tables/truth_scores_pooled.csv',
]


def complete_n87_runs(root):
    hits = []
    if root.exists():
        for path in root.iterdir():
            if path.is_dir() and all((path / rel).exists() for rel in N87_REQUIRED):
                hits.append(path)
    return sorted(hits, key=lambda p: p.stat().st_mtime)


if SOURCE_N87_RUN_OVERRIDE is not None:
    SOURCE_N87 = Path(SOURCE_N87_RUN_OVERRIDE)
    missing = [rel for rel in N87_REQUIRED if not (SOURCE_N87 / rel).exists()]
    require(not missing, f'Notebook 87 source override is incomplete: {missing}')
else:
    runs = complete_n87_runs(N87_ROOT)
    require(runs, f'No complete Notebook 87 run found below {N87_ROOT}')
    SOURCE_N87 = runs[-1]

prediction_lock = json.loads((SOURCE_N87 / 'prediction_lock.json').read_text())
verdict87 = json.loads((SOURCE_N87 / 'notebook87_verdict.json').read_text())

require(
    prediction_lock.get('state') == 'CLOSED_BEFORE_CLEAN_TRUTH',
    'Notebook 87 prediction lock is not closed before clean truth.',
)
require(
    prediction_lock.get('clean_truth_read') is False,
    'Notebook 87 prediction lock reports truth leakage.',
)
require(
    verdict87.get('automatic_promotion') is False,
    'Unexpected Notebook 87 promotion state.',
)

review_manifest_path = ROOT / 'paired_hitl_development' / 'locks' / 'paired_hitl_review_manifest.csv'
review_lock_path = ROOT / 'paired_hitl_development' / 'locks' / 'paired_hitl_review_lock.json'
require(review_manifest_path.exists(), f'Missing review manifest: {review_manifest_path}')
require(review_lock_path.exists(), f'Missing review lock: {review_lock_path}')

review_manifest = pd.read_csv(review_manifest_path, dtype=str).fillna('')
truth_pixels = pd.read_parquet(SOURCE_N87 / 'tables' / 'paired_hitl_ambiguous_truth.parquet')
truth_scores87 = pd.read_csv(SOURCE_N87 / 'tables' / 'truth_scores_by_page.csv')

required_truth = {
    'page', 'y', 'x', 'p_annotation', 'p_print',
    'truth_print', 'truth_annotation', 'truth_uncertain',
}
require(required_truth.issubset(truth_pixels.columns),
        f'Notebook 87 truth table missing: {sorted(required_truth-set(truth_pixels.columns))}')
require(not truth_pixels.duplicated(['page','y','x']).any(),
        'Notebook 87 truth table has duplicate page/y/x rows.')

# Snapshot the exact source records used by this run.
for source, target in [
    (SOURCE_N87 / 'prediction_lock.json', INPUTS / 'notebook87_prediction_lock.json'),
    (SOURCE_N87 / 'notebook87_verdict.json', INPUTS / 'notebook87_verdict.json'),
    (review_manifest_path, INPUTS / 'paired_hitl_review_manifest.csv'),
    (review_lock_path, INPUTS / 'paired_hitl_review_lock.json'),
]:
    shutil.copy2(source, target)

print('Notebook 87 source:', SOURCE_N87)
print('Truth rows:', len(truth_pixels))
print('Truth pages:', truth_pixels.page.astype(str).nunique())

# 3. Resolve page roles and build a fail-closed source-group map

In [ ]:
# @title
review_manifest['page'] = review_manifest.page.astype(str)
truth_pixels['page'] = truth_pixels.page.astype(str)

if 'source_group' not in review_manifest.columns:
    review_manifest['source_group'] = ''

# Missing source groups are not allowed to leak pages across folds.
# Each missing group becomes a page-specific group, which is conservative.
review_manifest['source_group'] = review_manifest.apply(
    lambda r: str(r.source_group).strip() if str(r.source_group).strip()
    else f'PAGE_ONLY__{r.page}',
    axis=1,
)

page_meta = review_manifest.set_index('page')
truth_pages = sorted(truth_pixels.page.unique())
missing_pages = sorted(set(truth_pages) - set(page_meta.index))
require(not missing_pages, f'Truth pages absent from review manifest: {missing_pages}')

# Pages with no eligible truth rows do not enter model fitting.
role_rows = []
for page in review_manifest.page:
    n = int((truth_pixels.page == page).sum())
    role_rows.append({
        'page': page,
        'source_group': page_meta.loc[page, 'source_group'],
        'eligible_truth_rows': n,
        'model_role': 'DEVELOPMENT_OOF' if n > 0 else 'ZERO_EXPOSURE_CONTROL',
    })
roles = pd.DataFrame(role_rows)
roles.to_csv(INPUTS / 'page_role_manifest.csv', index=False)

print(roles.model_role.value_counts())
print('Development source groups:', roles.loc[roles.model_role=='DEVELOPMENT_OOF','source_group'].nunique())

# 4. Observable channel construction

MEMORY-SAFE V3 WITH EXPLICIT SPARSE p_print REPRESENTATION

Clean truth is not loaded here. Labels come only from the
already post-lock Notebook 87 truth table.

p_print is sparse in Notebook 87:

  p_print_value:
      finite p_print probability where available;
      zero where no probability was supplied.

  p_print_available:
      1 where p_print_value represents a supplied probability;
      0 where the source p_print was non-finite/unavailable.

Each page is constructed separately, written to .npy files,
reopened read-only through NumPy memory maps, and released
before the next page is built.

In [ ]:
# @title


import gc
import json
import resource
import shutil

from pathlib import Path

import cv2
import numpy as np
import pandas as pd


# ============================================================
# Schema and memory configuration
# ============================================================

CELL4_CACHE_VERSION = (
    "N88_CELL4_MEMORY_SAFE_V3_SPARSE_PPRINT"
)


EXPECTED_TENSOR_CHANNELS = [
    "observed",
    "local_normalised",
    "reviewed_mask",
    "mask_depth",
    "ambiguous",
    "p_annotation",
    "p_print_value",
    "p_print_available",
    "edge",
    "horizontal_support",
    "vertical_support",
    "compact_support",
]


require(
    list(CHANNELS)
    == EXPECTED_TENSOR_CHANNELS,
    (
        "CHANNELS does not match the Cell 4 V3 tensor schema.\n"
        f"Expected: {EXPECTED_TENSOR_CHANNELS}\n"
        f"Actual:   {list(CHANNELS)}"
    ),
)


CHANNEL_CACHE = (
    RUN
    / "observable_channel_cache"
)


CHANNEL_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)


# False:
#   reuse a complete, hash-compatible cache.
#
# True:
#   rebuild every page regardless of cache state.
FORCE_CHANNEL_REBUILD = False


REPORT_MEMORY = True


try:
    cv2.setNumThreads(1)
except Exception:
    pass


try:
    cv2.ocl.setUseOpenCL(False)
except Exception:
    pass


# Remove stale runtime memory maps from a previous execution.
if "runtime" in globals():

    try:
        runtime.clear()
    except Exception:
        pass

    del runtime


gc.collect()


if torch.cuda.is_available():
    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass


# ============================================================
# Cached channel schema
#
# A1 and paper are retained for later page assembly, but are not
# CNN input channels.
# ============================================================

CACHED_CHANNEL_DTYPES = {
    "observed":
        np.uint8,

    "a1":
        np.uint8,

    "paper":
        np.uint8,

    "reviewed_mask":
        np.uint8,

    "ambiguous":
        np.uint8,

    "local_normalised":
        np.uint8,

    "mask_depth":
        np.float32,

    "p_annotation":
        np.float32,

    "p_print_value":
        np.float32,

    "p_print_available":
        np.uint8,

    "edge":
        np.uint8,

    "horizontal_support":
        np.uint8,

    "vertical_support":
        np.uint8,

    "compact_support":
        np.uint8,
}


require(
    set(CHANNELS)
    <= set(CACHED_CHANNEL_DTYPES),
    (
        "Tensor channels missing from the cache schema: "
        f"{sorted(set(CHANNELS) - set(CACHED_CHANNEL_DTYPES))}"
    ),
)


# ============================================================
# Basic helpers
# ============================================================

def read_gray(path):

    image = cv2.imread(
        str(path),
        cv2.IMREAD_GRAYSCALE,
    )

    require(
        image is not None,
        f"Unable to read image: {path}",
    )

    require(
        image.ndim == 2,
        f"Image is not single-channel: {path}",
    )

    return image


def reconcile_zero_authority_canvas(
    array,
    target_shape,
    page,
    name,
):

    if array.shape == target_shape:
        return array

    h, w = array.shape
    target_h, target_w = target_shape

    delta_h = target_h - h
    delta_w = target_w - w

    require(
        delta_h in (0, 1)
        and delta_w in (0, 1),
        (
            f"{page}: {name} shape {array.shape} "
            f"cannot safely reconcile to {target_shape}"
        ),
    )

    fixed = np.zeros(
        target_shape,
        dtype=array.dtype,
    )

    fixed[
        :h,
        :w,
    ] = array

    return fixed


def process_memory_gib():

    # Linux/Colab ru_maxrss is reported in KiB.
    return (
        resource.getrusage(
            resource.RUSAGE_SELF
        ).ru_maxrss
        / 1024**2
    )


def channel_path(
    page,
    channel,
):

    return (
        CHANNEL_CACHE
        / str(page)
        / f"{channel}.npy"
    )


def page_cache_manifest_path(page):

    return (
        CHANNEL_CACHE
        / str(page)
        / "cache_manifest.json"
    )


def atomic_save_npy(
    path,
    array,
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    with open(
        temporary,
        "wb",
    ) as handle:

        np.save(
            handle,
            array,
            allow_pickle=False,
        )

    temporary.replace(path)


def atomic_write_json(
    path,
    value,
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            default=str,
        )
    )

    temporary.replace(path)


def load_mmap(path):

    return np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )


def array_record(path):

    path = Path(path)

    array = load_mmap(path)

    record = {
        "path":
            str(path),

        "shape":
            list(array.shape),

        "dtype":
            str(array.dtype),

        "bytes":
            int(path.stat().st_size),

        "sha256":
            sha256_file(path),
    }

    del array

    return record


# ============================================================
# Memory-reduced paper/background surface
# ============================================================

def background_surface(
    observed,
    mask,
    sigma=45.0,
):

    observed = np.asarray(
        observed,
        dtype=np.uint8,
    )

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    visible_u8 = (
        (
            ~mask
        )
        & (
            observed
            >= 200
        )
    ).astype(
        np.uint8
    )

    weighted = observed.astype(
        np.float32
    )

    weighted *= visible_u8

    numerator = cv2.GaussianBlur(
        weighted,
        (0, 0),
        sigma,
    )

    del weighted

    denominator_input = visible_u8.astype(
        np.float32
    )

    denominator = cv2.GaussianBlur(
        denominator_input,
        (0, 0),
        sigma,
    )

    del denominator_input
    del visible_u8

    np.divide(
        numerator,
        np.maximum(
            denominator,
            1e-4,
        ),
        out=numerator,
    )

    fallback = cv2.GaussianBlur(
        observed,
        (0, 0),
        sigma,
    )

    weak = (
        denominator
        <= 0.02
    )

    numerator[
        weak
    ] = fallback[
        weak
    ]

    paper = np.clip(
        numerator,
        0,
        255,
    ).astype(
        np.uint8
    )

    del numerator
    del denominator
    del fallback
    del weak

    return paper


# ============================================================
# Memory-reduced support maps
# ============================================================

def support_maps(
    observed,
    mask,
    band=4,
):

    observed = np.asarray(
        observed,
        dtype=np.uint8,
    )

    mask = np.asarray(
        mask,
        dtype=bool,
    )

    dark_u8 = np.where(
        (
            observed
            < 190
        )
        & (
            ~mask
        ),
        255,
        0,
    ).astype(
        np.uint8
    )

    horizontal_u8 = cv2.morphologyEx(
        dark_u8,
        cv2.MORPH_OPEN,
        np.ones(
            (1, 35),
            dtype=np.uint8,
        ),
    )

    vertical_u8 = cv2.morphologyEx(
        dark_u8,
        cv2.MORPH_OPEN,
        np.ones(
            (15, 1),
            dtype=np.uint8,
        ),
    )

    (
        component_count,
        labels,
        stats,
        _,
    ) = cv2.connectedComponentsWithStats(
        (
            dark_u8
            > 0
        ).astype(
            np.uint8
        ),
        8,
    )

    compact_component = np.zeros(
        component_count,
        dtype=np.uint8,
    )

    if component_count > 1:

        widths = stats[
            :,
            cv2.CC_STAT_WIDTH,
        ]

        heights = stats[
            :,
            cv2.CC_STAT_HEIGHT,
        ]

        areas = stats[
            :,
            cv2.CC_STAT_AREA,
        ]

        compact_component[
            (
                areas
                >= 3
            )
            & (
                areas
                <= 650
            )
            & (
                widths
                <= 40
            )
            & (
                heights
                <= 40
            )
        ] = 1

        compact_component[0] = 0

    compact_u8 = compact_component[
        labels
    ]

    del labels
    del stats
    del compact_component
    del dark_u8

    kernel = np.ones(
        (
            2 * band + 1,
            2 * band + 1,
        ),
        dtype=np.uint8,
    )

    horizontal = cv2.dilate(
        (
            horizontal_u8
            > 0
        ).astype(
            np.uint8
        ),
        kernel,
    )

    del horizontal_u8

    vertical = cv2.dilate(
        (
            vertical_u8
            > 0
        ).astype(
            np.uint8
        ),
        kernel,
    )

    del vertical_u8

    compact = cv2.dilate(
        compact_u8,
        kernel,
    )

    del compact_u8
    del kernel

    return (
        horizontal,
        vertical,
        compact,
    )


# ============================================================
# Cache identity and validation
# ============================================================

def expected_page_fingerprint(page_dir):

    return {
        "observed_sha256":
            sha256_file(
                page_dir
                / "observed.png"
            ),

        "reviewed_seed_sha256":
            sha256_file(
                page_dir
                / "reviewed_seed.png"
            ),

        "ambiguous_sha256":
            sha256_file(
                page_dir
                / "ambiguous.png"
            ),

        "a1_sha256":
            sha256_file(
                page_dir
                / "A1_BASELINE.png"
            ),

        "arrays_sha256":
            sha256_file(
                page_dir
                / "arrays.npz"
            ),

        "channel_build_version":
            CELL4_CACHE_VERSION,

        "ordered_tensor_channels":
            list(CHANNELS),
    }


def page_cache_is_valid(
    page,
    page_dir,
):

    manifest_path = page_cache_manifest_path(
        page
    )

    if (
        FORCE_CHANNEL_REBUILD
        or not manifest_path.exists()
    ):
        return False

    try:

        record = json.loads(
            manifest_path.read_text()
        )

        if (
            record.get("status")
            != "COMPLETE"
        ):
            return False

        if (
            record.get("fingerprint")
            != expected_page_fingerprint(
                page_dir
            )
        ):
            return False

        shape = tuple(
            record["shape"]
        )

        for channel, dtype in CACHED_CHANNEL_DTYPES.items():

            path = channel_path(
                page,
                channel,
            )

            if not path.exists():
                return False

            array = load_mmap(path)

            valid = (
                array.ndim
                == 2
                and array.shape
                == shape
                and array.dtype
                == np.dtype(dtype)
            )

            del array

            if not valid:
                return False

        return True

    except Exception:
        return False


# ============================================================
# Build and persist one page
# ============================================================

def build_cached_page(
    page,
    page_dir,
    truth_rows,
):

    observed = read_gray(
        page_dir
        / "observed.png"
    )

    seed = read_gray(
        page_dir
        / "reviewed_seed.png"
    )

    ambiguous = (
        read_gray(
            page_dir
            / "ambiguous.png"
        )
        > 0
    )

    a1 = read_gray(
        page_dir
        / "A1_BASELINE.png"
    )

    seed_original_shape = seed.shape

    seed = reconcile_zero_authority_canvas(
        seed,
        observed.shape,
        page,
        "reviewed seed",
    )

    repair = None

    if seed.shape != seed_original_shape:

        repair = {
            "page":
                page,

            "original_shape":
                str(seed_original_shape),

            "target_shape":
                str(observed.shape),

            "authority_pixels_added":
                0,
        }

    # --------------------------------------------------------
    # Load everything needed from arrays.npz while the context
    # is still open.
    # --------------------------------------------------------

    with np.load(
        page_dir
        / "arrays.npz",
        allow_pickle=False,
    ) as source_arrays:

        required_npz_arrays = {
            "p_annotation",
            "p_print",
            "ambiguous",
        }

        missing_npz_arrays = (
            required_npz_arrays
            - set(source_arrays.files)
        )

        require(
            not missing_npz_arrays,
            (
                f"{page}: arrays.npz missing "
                f"{sorted(missing_npz_arrays)}"
            ),
        )

        p_annotation = np.array(
            source_arrays[
                "p_annotation"
            ],
            dtype=np.float32,
            copy=True,
        )

        p_print_source = np.array(
            source_arrays[
                "p_print"
            ],
            dtype=np.float32,
            copy=True,
        )

        ambiguous_npz = (
            np.array(
                source_arrays[
                    "ambiguous"
                ],
                copy=True,
            )
            > 0
        )

    # --------------------------------------------------------
    # Shape integrity
    # --------------------------------------------------------

    require(
        observed.ndim == 2,
        f"{page}: observed array is not 2D",
    )

    require(
        (
            observed.shape
            == seed.shape
            == ambiguous.shape
            == a1.shape
        ),
        (
            f"{page}: observable image shape mismatch: "
            f"observed={observed.shape}, "
            f"seed={seed.shape}, "
            f"ambiguous={ambiguous.shape}, "
            f"A1={a1.shape}"
        ),
    )

    require(
        (
            p_annotation.shape
            == observed.shape
        )
        and (
            p_print_source.shape
            == observed.shape
        )
        and (
            ambiguous_npz.shape
            == observed.shape
        ),
        (
            f"{page}: NPZ array shape mismatch: "
            f"observed={observed.shape}, "
            f"p_annotation={p_annotation.shape}, "
            f"p_print={p_print_source.shape}, "
            f"ambiguous_npz={ambiguous_npz.shape}"
        ),
    )

    require(
        np.array_equal(
            ambiguous,
            ambiguous_npz,
        ),
        (
            f"{page}: ambiguous image/NPZ mismatch"
        ),
    )

    # --------------------------------------------------------
    # Dense annotation probability
    # --------------------------------------------------------

    require(
        np.isfinite(
            p_annotation
        ).all(),
        (
            f"{page}: p_annotation contains "
            "non-finite values"
        ),
    )

    require(
        (
            (
                p_annotation
                >= 0.0
            )
            & (
                p_annotation
                <= 1.0
            )
        ).all(),
        (
            f"{page}: p_annotation contains "
            "values outside [0,1]"
        ),
    )

    # --------------------------------------------------------
    # Sparse p_print representation
    # --------------------------------------------------------

    p_print_available = np.isfinite(
        p_print_source
    ).astype(
        np.uint8
    )

    valid_p_print = (
        p_print_available
        > 0
    )

    p_print_value = np.zeros(
        observed.shape,
        dtype=np.float32,
    )

    p_print_value[
        valid_p_print
    ] = p_print_source[
        valid_p_print
    ]

    p_print_available_count = int(
        valid_p_print.sum()
    )

    p_print_unavailable_count = int(
        valid_p_print.size
        - p_print_available_count
    )

    require(
        np.isfinite(
            p_print_value
        ).all(),
        (
            f"{page}: p_print_value contains "
            "non-finite values"
        ),
    )

    if p_print_available_count:

        require(
            (
                (
                    p_print_value[
                        valid_p_print
                    ]
                    >= 0.0
                )
                & (
                    p_print_value[
                        valid_p_print
                    ]
                    <= 1.0
                )
            ).all(),
            (
                f"{page}: available p_print values "
                "fall outside [0,1]"
            ),
        )

    require(
        set(
            np.unique(
                p_print_available
            ).tolist()
        )
        <= {
            0,
            1,
        },
        (
            f"{page}: p_print_available is not binary"
        ),
    )

    reviewed_mask = (
        seed
        > 0
    )

    # --------------------------------------------------------
    # Truth-coordinate integrity
    # --------------------------------------------------------

    ys = truth_rows.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = truth_rows.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    require(
        (
            (
                ys
                >= 0
            ).all()
            and (
                xs
                >= 0
            ).all()
            and (
                ys
                < observed.shape[0]
            ).all()
            and (
                xs
                < observed.shape[1]
            ).all()
        ),
        (
            f"{page}: truth coordinate outside "
            "observable arrays"
        ),
    )

    require(
        ambiguous[
            ys,
            xs,
        ].all(),
        (
            f"{page}: truth table contains "
            "non-ambiguous coordinates"
        ),
    )

    p_print_available_at_truth = (
        p_print_available[
            ys,
            xs,
        ]
        > 0
    )

    p_print_truth_available_count = int(
        p_print_available_at_truth.sum()
    )

    p_print_truth_unavailable_count = int(
        len(truth_rows)
        - p_print_truth_available_count
    )

    p_print_truth_availability_rate = float(
        p_print_available_at_truth.mean()
    )

    # --------------------------------------------------------
    # Rebuild page cache atomically at page-directory level
    # --------------------------------------------------------

    page_cache_dir = (
        CHANNEL_CACHE
        / page
    )

    if page_cache_dir.exists():
        shutil.rmtree(page_cache_dir)

    page_cache_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    # --------------------------------------------------------
    # Write source channels
    # --------------------------------------------------------

    atomic_save_npy(
        channel_path(
            page,
            "observed",
        ),
        observed.astype(
            np.uint8,
            copy=False,
        ),
    )

    atomic_save_npy(
        channel_path(
            page,
            "a1",
        ),
        a1.astype(
            np.uint8,
            copy=False,
        ),
    )

    atomic_save_npy(
        channel_path(
            page,
            "reviewed_mask",
        ),
        reviewed_mask.astype(
            np.uint8,
            copy=False,
        ),
    )

    atomic_save_npy(
        channel_path(
            page,
            "ambiguous",
        ),
        ambiguous.astype(
            np.uint8,
            copy=False,
        ),
    )

    atomic_save_npy(
        channel_path(
            page,
            "p_annotation",
        ),
        p_annotation,
    )

    atomic_save_npy(
        channel_path(
            page,
            "p_print_value",
        ),
        p_print_value,
    )

    atomic_save_npy(
        channel_path(
            page,
            "p_print_available",
        ),
        p_print_available,
    )

    del seed
    del a1
    del ambiguous_npz
    del p_annotation
    del p_print_source
    del p_print_value
    del p_print_available
    del valid_p_print
    del p_print_available_at_truth

    gc.collect()

    # --------------------------------------------------------
    # Paper and local normalisation
    # --------------------------------------------------------

    paper = background_surface(
        observed,
        reviewed_mask,
    )

    atomic_save_npy(
        channel_path(
            page,
            "paper",
        ),
        paper,
    )

    local_normalised = cv2.divide(
        observed,
        np.maximum(
            paper,
            1,
        ),
        scale=255,
    )

    atomic_save_npy(
        channel_path(
            page,
            "local_normalised",
        ),
        local_normalised.astype(
            np.uint8,
            copy=False,
        ),
    )

    del paper
    del local_normalised

    gc.collect()

    # --------------------------------------------------------
    # Mask depth
    # --------------------------------------------------------

    mask_depth = cv2.distanceTransform(
        reviewed_mask.astype(
            np.uint8
        ),
        cv2.DIST_L2,
        5,
    ).astype(
        np.float32,
        copy=False,
    )

    require(
        np.isfinite(
            mask_depth
        ).all(),
        (
            f"{page}: mask_depth contains "
            "non-finite values"
        ),
    )

    atomic_save_npy(
        channel_path(
            page,
            "mask_depth",
        ),
        mask_depth,
    )

    del mask_depth

    gc.collect()

    # --------------------------------------------------------
    # Edge channel
    # --------------------------------------------------------

    edge = (
        cv2.Canny(
            observed,
            70,
            170,
        )
        > 0
    ).astype(
        np.uint8
    )

    atomic_save_npy(
        channel_path(
            page,
            "edge",
        ),
        edge,
    )

    del edge

    gc.collect()

    # --------------------------------------------------------
    # Support channels
    # --------------------------------------------------------

    (
        horizontal,
        vertical,
        compact,
    ) = support_maps(
        observed,
        reviewed_mask,
    )

    atomic_save_npy(
        channel_path(
            page,
            "horizontal_support",
        ),
        horizontal.astype(
            np.uint8,
            copy=False,
        ),
    )

    del horizontal
    gc.collect()

    atomic_save_npy(
        channel_path(
            page,
            "vertical_support",
        ),
        vertical.astype(
            np.uint8,
            copy=False,
        ),
    )

    del vertical
    gc.collect()

    atomic_save_npy(
        channel_path(
            page,
            "compact_support",
        ),
        compact.astype(
            np.uint8,
            copy=False,
        ),
    )

    del compact
    gc.collect()

    # --------------------------------------------------------
    # Record every persisted channel
    # --------------------------------------------------------

    channel_records = {
        channel:
            array_record(
                channel_path(
                    page,
                    channel,
                )
            )

        for channel
        in CACHED_CHANNEL_DTYPES
    }

    cache_record = {
        "page":
            page,

        "status":
            "COMPLETE",

        "shape":
            list(observed.shape),

        "fingerprint":
            expected_page_fingerprint(
                page_dir
            ),

        "ordered_tensor_channels":
            list(CHANNELS),

        "channels":
            channel_records,

        "eligible_rows":
            int(len(truth_rows)),

        "sparse_p_print": {
            "source_semantics":
                (
                    "Finite only where Notebook 87 supplied "
                    "an observed-print probability"
                ),

            "value_channel":
                "p_print_value",

            "availability_channel":
                "p_print_available",

            "unavailable_value_fill":
                0.0,

            "available_page_pixels":
                p_print_available_count,

            "unavailable_page_pixels":
                p_print_unavailable_count,

            "page_availability_rate":
                float(
                    p_print_available_count
                    / max(
                        observed.size,
                        1,
                    )
                ),

            "available_truth_centres":
                p_print_truth_available_count,

            "unavailable_truth_centres":
                p_print_truth_unavailable_count,

            "truth_centre_availability_rate":
                p_print_truth_availability_rate,
        },
    }

    atomic_write_json(
        page_cache_manifest_path(
            page
        ),
        cache_record,
    )

    shape = observed.shape

    del observed
    del reviewed_mask
    del ambiguous
    del ys
    del xs
    del channel_records

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        shape,
        repair,
    )


# ============================================================
# Lightweight disk-backed runtime
# ============================================================

class DiskBackedPage(dict):

    def __init__(
        self,
        page,
    ):

        super().__init__()

        self.page = str(page)

        for channel in CACHED_CHANNEL_DTYPES:

            self[channel] = load_mmap(
                channel_path(
                    self.page,
                    channel,
                )
            )


# ============================================================
# Build or reuse all pages
# ============================================================

runtime = {}

input_rows = []

canvas_repairs = []

cache_rows = []

p_print_availability_rows = []


for page_number, page in enumerate(
    truth_pages,
    start=1,
):

    page = str(page)

    row = page_meta.loc[page]

    page_dir = (
        SOURCE_N87
        / "pages"
        / page
    )

    require(
        page_dir.exists(),
        (
            f"{page}: Notebook 87 page "
            "directory missing"
        ),
    )

    required_page_files = [
        page_dir
        / "observed.png",

        page_dir
        / "reviewed_seed.png",

        page_dir
        / "ambiguous.png",

        page_dir
        / "A1_BASELINE.png",

        page_dir
        / "arrays.npz",
    ]

    missing_page_files = [
        str(path)

        for path
        in required_page_files

        if not path.exists()
    ]

    require(
        not missing_page_files,
        (
            f"{page}: missing Notebook 87 "
            f"artefacts: {missing_page_files}"
        ),
    )

    q = truth_pixels[
        truth_pixels.page.astype(str)
        == page
    ].copy()

    require(
        len(q) > 0,
        (
            f"{page}: no eligible truth rows "
            "available"
        ),
    )

    reused = page_cache_is_valid(
        page,
        page_dir,
    )

    repair = None

    if reused:

        shape = tuple(
            json.loads(
                page_cache_manifest_path(
                    page
                ).read_text()
            )["shape"]
        )

        status = "REUSED"

    else:

        shape, repair = build_cached_page(
            page,
            page_dir,
            q,
        )

        status = "BUILT"

    if repair is not None:
        canvas_repairs.append(repair)

    cache_manifest = json.loads(
        page_cache_manifest_path(
            page
        ).read_text()
    )

    runtime[page] = DiskBackedPage(page)

    # --------------------------------------------------------
    # Read-only cached-array integrity
    # --------------------------------------------------------

    for channel, array in runtime[
        page
    ].items():

        require(
            array.ndim == 2,
            (
                f"{page}: cached channel "
                f"{channel} is not 2D"
            ),
        )

        require(
            array.shape == shape,
            (
                f"{page}: cached channel "
                f"{channel} shape {array.shape} "
                f"!= {shape}"
            ),
        )

        require(
            array.dtype
            == np.dtype(
                CACHED_CHANNEL_DTYPES[
                    channel
                ]
            ),
            (
                f"{page}: cached channel "
                f"{channel} dtype {array.dtype} "
                f"!= "
                f"{np.dtype(CACHED_CHANNEL_DTYPES[channel])}"
            ),
        )

    ys = q.y.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    xs = q.x.to_numpy(
        dtype=np.int32,
        copy=False,
    )

    require(
        (
            runtime[
                page
            ][
                "ambiguous"
            ][
                ys,
                xs,
            ]
            > 0
        ).all(),
        (
            f"{page}: disk-backed ambiguity channel "
            "does not contain every truth coordinate"
        ),
    )

    centre_available = (
        runtime[
            page
        ][
            "p_print_available"
        ][
            ys,
            xs,
        ]
        > 0
    )

    sparse_record = cache_manifest[
        "sparse_p_print"
    ]

    require(
        int(
            centre_available.sum()
        )
        == int(
            sparse_record[
                "available_truth_centres"
            ]
        ),
        (
            f"{page}: cached p_print centre-availability "
            "count does not match manifest"
        ),
    )

    input_rows.append({
        "page":
            page,

        "source_group":
            row.source_group,

        "height":
            int(shape[0]),

        "width":
            int(shape[1]),

        "eligible_rows":
            int(len(q)),

        "truth_print":
            int(
                q.truth_print.astype(
                    bool
                ).sum()
            ),

        "truth_annotation":
            int(
                q.truth_annotation.astype(
                    bool
                ).sum()
            ),

        "truth_uncertain":
            int(
                q.truth_uncertain.astype(
                    bool
                ).sum()
            ),

        "p_print_available_page_pixels":
            int(
                sparse_record[
                    "available_page_pixels"
                ]
            ),

        "p_print_unavailable_page_pixels":
            int(
                sparse_record[
                    "unavailable_page_pixels"
                ]
            ),

        "p_print_page_availability_rate":
            float(
                sparse_record[
                    "page_availability_rate"
                ]
            ),

        "p_print_available_truth_centres":
            int(
                sparse_record[
                    "available_truth_centres"
                ]
            ),

        "p_print_unavailable_truth_centres":
            int(
                sparse_record[
                    "unavailable_truth_centres"
                ]
            ),

        "p_print_truth_centre_availability_rate":
            float(
                sparse_record[
                    "truth_centre_availability_rate"
                ]
            ),

        "observed_sha256":
            sha256_file(
                page_dir
                / "observed.png"
            ),

        "arrays_sha256":
            sha256_file(
                page_dir
                / "arrays.npz"
            ),

        "a1_sha256":
            sha256_file(
                page_dir
                / "A1_BASELINE.png"
            ),

        "channel_cache_manifest":
            str(
                page_cache_manifest_path(
                    page
                )
            ),

        "channel_cache_manifest_sha256":
            sha256_file(
                page_cache_manifest_path(
                    page
                )
            ),

        "channel_cache_status":
            status,
    })

    p_print_availability_rows.append({
        "page":
            page,

        "source_group":
            row.source_group,

        "page_pixels":
            int(
                shape[0]
                * shape[1]
            ),

        "available_page_pixels":
            int(
                sparse_record[
                    "available_page_pixels"
                ]
            ),

        "unavailable_page_pixels":
            int(
                sparse_record[
                    "unavailable_page_pixels"
                ]
            ),

        "page_availability_rate":
            float(
                sparse_record[
                    "page_availability_rate"
                ]
            ),

        "eligible_truth_centres":
            int(len(q)),

        "available_truth_centres":
            int(
                centre_available.sum()
            ),

        "unavailable_truth_centres":
            int(
                (
                    ~centre_available
                ).sum()
            ),

        "truth_centre_availability_rate":
            float(
                centre_available.mean()
            ),
    })

    cache_bytes = sum(
        channel_path(
            page,
            channel,
        ).stat().st_size

        for channel
        in CACHED_CHANNEL_DTYPES
    )

    cache_rows.append({
        "page":
            page,

        "status":
            status,

        "height":
            int(shape[0]),

        "width":
            int(shape[1]),

        "cached_channels":
            int(
                len(
                    CACHED_CHANNEL_DTYPES
                )
            ),

        "tensor_channels":
            int(
                len(CHANNELS)
            ),

        "cache_bytes":
            int(cache_bytes),

        "cache_gib":
            float(
                cache_bytes
                / 1024**3
            ),

        "peak_process_memory_gib":
            float(
                process_memory_gib()
            ),
    })

    del q
    del ys
    del xs
    del centre_available
    del cache_manifest
    del sparse_record

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if REPORT_MEMORY:

        print(
            f"[{page_number:03d}/"
            f"{len(truth_pages):03d}] "
            f"{page}: {status} | "
            f"shape={shape} | "
            f"disk={cache_bytes / 1024**2:.1f} MiB | "
            f"peak RAM={process_memory_gib():.2f} GiB"
        )

    else:

        print(
            f"[{page_number:03d}/"
            f"{len(truth_pages):03d}] "
            f"{page}: {status}"
        )


# ============================================================
# Audit outputs
# ============================================================

input_audit = pd.DataFrame(
    input_rows
)


input_audit.to_csv(
    AUDIT
    / "input_and_label_audit.csv",
    index=False,
)


pd.DataFrame(
    canvas_repairs,
    columns=[
        "page",
        "original_shape",
        "target_shape",
        "authority_pixels_added",
    ],
).to_csv(
    AUDIT
    / "reviewed_seed_canvas_repairs.csv",
    index=False,
)


cache_audit = pd.DataFrame(
    cache_rows
)


cache_audit.to_csv(
    AUDIT
    / "observable_channel_cache_audit.csv",
    index=False,
)


p_print_availability_audit = pd.DataFrame(
    p_print_availability_rows
)


p_print_availability_audit.to_csv(
    AUDIT
    / "p_print_availability_audit.csv",
    index=False,
)


# ============================================================
# Channel schema
# ============================================================

channel_schema = {
    "schema_version":
        CELL4_CACHE_VERSION,

    "patch":
        PATCH,

    "ordered_channels":
        list(CHANNELS),

    "tensor_channel_count":
        len(CHANNELS),

    "tensor_dtype":
        "float32_at_patch_extraction",

    "cached_channel_dtypes": {
        channel:
            str(
                np.dtype(dtype)
            )

        for channel, dtype
        in CACHED_CHANNEL_DTYPES.items()
    },

    "normalisation": {
        "observed":
            "divide_by_255_at_patch_extraction",

        "local_normalised":
            "divide_by_255_at_patch_extraction",

        "reviewed_mask":
            "cached_uint8_0_or_1_then_float32",

        "mask_depth":
            (
                "divide_by_patch_radius_and_clip_0_1_"
                "at_patch_extraction"
            ),

        "ambiguous":
            "cached_uint8_0_or_1_then_float32",

        "p_annotation":
            "native_float32_0_to_1",

        "p_print_value":
            (
                "native_float32_0_to_1_where_"
                "p_print_available_is_1; otherwise_0"
            ),

        "p_print_available":
            (
                "binary_1_where_source_p_print_is_finite_"
                "else_0"
            ),

        "edge":
            "cached_uint8_0_or_1_then_float32",

        "horizontal_support":
            "cached_uint8_0_or_1_then_float32",

        "vertical_support":
            "cached_uint8_0_or_1_then_float32",

        "compact_support":
            "cached_uint8_0_or_1_then_float32",
    },

    "binary_tensor_channels": [
        "reviewed_mask",
        "ambiguous",
        "p_print_available",
        "edge",
        "horizontal_support",
        "vertical_support",
        "compact_support",
    ],

    "probability_tensor_channels": [
        "p_annotation",
        "p_print_value",
    ],

    "sparse_probability_representation": {
        "source":
            "p_print",

        "value_channel":
            "p_print_value",

        "availability_channel":
            "p_print_available",

        "unavailable_value_fill":
            0.0,

        "interpretation":
            (
                "A zero p_print_value with availability zero "
                "means unavailable, not zero print probability."
            ),
    },

    "padding":
        (
            "reflect-equivalent coordinate mapping "
            "during on-demand patch extraction"
        ),

    "storage": {
        "type":
            "per_page_npy_read_only_memmap",

        "root":
            str(CHANNEL_CACHE),

        "cache_version":
            CELL4_CACHE_VERSION,
    },

    "truth_derived_channels":
        [],
}


(
    INPUTS
    / "channel_schema.json"
).write_text(
    json.dumps(
        channel_schema,
        indent=2,
    )
)


# ============================================================
# Final integrity and availability report
# ============================================================

require(
    len(runtime)
    == len(truth_pages),
    (
        "Disk-backed runtime does not cover "
        "the complete truth-page roster"
    ),
)


totals = input_audit[
    [
        "eligible_rows",
        "truth_print",
        "truth_annotation",
        "truth_uncertain",
    ]
].sum()


availability_totals = (
    p_print_availability_audit[
        [
            "page_pixels",
            "available_page_pixels",
            "unavailable_page_pixels",
            "eligible_truth_centres",
            "available_truth_centres",
            "unavailable_truth_centres",
        ]
    ]
    .sum()
)


overall_page_availability = float(
    availability_totals[
        "available_page_pixels"
    ]
    / max(
        availability_totals[
            "page_pixels"
        ],
        1,
    )
)


overall_truth_centre_availability = float(
    availability_totals[
        "available_truth_centres"
    ]
    / max(
        availability_totals[
            "eligible_truth_centres"
        ],
        1,
    )
)


print()
print("=" * 72)
print("OBSERVABLE CHANNEL CACHE COMPLETE")
print("=" * 72)

print(totals)

print()

print(
    "Pages:",
    len(runtime),
)

print(
    "Built:",
    int(
        (
            cache_audit.status
            == "BUILT"
        ).sum()
    ),
)

print(
    "Reused:",
    int(
        (
            cache_audit.status
            == "REUSED"
        ).sum()
    ),
)

print(
    "Tensor channels:",
    len(CHANNELS),
)

print(
    "Cached arrays per page:",
    len(CACHED_CHANNEL_DTYPES),
)

print(
    "Total channel-cache size:",
    f"{cache_audit.cache_bytes.sum() / 1024**3:.2f} GiB",
)

print(
    "Peak process memory reported:",
    f"{process_memory_gib():.2f} GiB",
)

print()

print(
    "p_print available page pixels:",
    f"{int(availability_totals['available_page_pixels']):,}",
)

print(
    "p_print unavailable page pixels:",
    f"{int(availability_totals['unavailable_page_pixels']):,}",
)

print(
    "Overall page-pixel p_print availability:",
    f"{overall_page_availability:.6%}",
)

print()

print(
    "Eligible truth centres:",
    f"{int(availability_totals['eligible_truth_centres']):,}",
)

print(
    "p_print available at eligible truth centres:",
    f"{int(availability_totals['available_truth_centres']):,}",
)

print(
    "p_print unavailable at eligible truth centres:",
    f"{int(availability_totals['unavailable_truth_centres']):,}",
)

print(
    "Eligible-centre p_print availability:",
    f"{overall_truth_centre_availability:.6%}",
)

print()

print(
    "Input audit:",
    AUDIT
    / "input_and_label_audit.csv",
)

print(
    "p_print availability audit:",
    AUDIT
    / "p_print_availability_audit.csv",
)

print(
    "Cache audit:",
    AUDIT
    / "observable_channel_cache_audit.csv",
)

print(
    "Channel schema:",
    INPUTS
    / "channel_schema.json",
)

print(
    "Cache root:",
    CHANNEL_CACHE,
)

# 5. Build labels, complete eligible roster and stable page-balanced training sample

In [ ]:
# @title
import gc


# ------------------------------------------------------------
# Build the eligible roster using vectorised operations
# ------------------------------------------------------------

eligible = truth_pixels[
    [
        "page",
        "y",
        "x",
        "truth_print",
        "truth_annotation",
        "truth_uncertain",
        "p_annotation",
        "p_print",
    ]
].copy()


eligible[
    "page"
] = eligible[
    "page"
].astype(str)


eligible[
    "source_group"
] = eligible[
    "page"
].map(
    page_meta[
        "source_group"
    ].astype(str)
)


require(
    eligible[
        "source_group"
    ].notna().all(),
    "One or more eligible rows have no source group.",
)


# ------------------------------------------------------------
# One-hot truth integrity
# ------------------------------------------------------------

truth_flag_count = (
    eligible[
        [
            "truth_print",
            "truth_annotation",
            "truth_uncertain",
        ]
    ]
    .astype(bool)
    .sum(axis=1)
)


bad_truth = (
    truth_flag_count
    != 1
)


if bad_truth.any():

    bad = eligible.loc[
        bad_truth,
        [
            "page",
            "y",
            "x",
            "truth_print",
            "truth_annotation",
            "truth_uncertain",
        ],
    ].head(100)

    bad.to_csv(
        AUDIT
        / "non_one_hot_truth_rows.csv",
        index=False,
    )

    raise RuntimeError(
        (
            f"{int(bad_truth.sum())} truth rows are not "
            "one-hot. See non_one_hot_truth_rows.csv"
        )
    )


# ------------------------------------------------------------
# Binary supervised label
#
# PRINT       -> LABEL_PRINT
# ANNOTATION  -> LABEL_ANNOTATION
# UNCERTAIN   -> LABEL_UNCERTAIN
# ------------------------------------------------------------

eligible[
    "label"
] = np.select(
    [
        eligible[
            "truth_print"
        ].astype(bool),

        eligible[
            "truth_annotation"
        ].astype(bool),
    ],
    [
        LABEL_PRINT,
        LABEL_ANNOTATION,
    ],
    default=LABEL_UNCERTAIN,
).astype(
    np.int8
)


eligible = eligible.rename(
    columns={
        "p_annotation":
            "p_annotation_existing",

        "p_print":
            "p_print_existing",
    }
)


eligible = (
    eligible[
        [
            "page",
            "source_group",
            "y",
            "x",
            "label",
            "p_annotation_existing",
            "p_print_existing",
        ]
    ]
    .sort_values(
        [
            "source_group",
            "page",
            "y",
            "x",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


eligible[
    "candidate_id"
] = np.arange(
    len(eligible),
    dtype=np.int64,
)


del truth_flag_count
del bad_truth

gc.collect()


# ------------------------------------------------------------
# Stable supervised sample
#
# UNCERTAIN stays in the complete eligible roster but is not
# used in the supervised PRINT-versus-ANNOTATION loss.
# ------------------------------------------------------------

supervised = eligible[
    eligible[
        "label"
    ].isin(
        [
            LABEL_PRINT,
            LABEL_ANNOTATION,
        ]
    )
]


require(
    not supervised.empty,
    "No supervised PRINT/ANNOTATION rows exist.",
)


rng = np.random.default_rng(
    SEED
)


sampled_indices = []


for (
    page,
    label,
), index_values in supervised.groupby(
    [
        "page",
        "label",
    ],
    sort=True,
).groups.items():

    index_values = np.asarray(
        index_values,
        dtype=np.int64,
    )

    if (
        len(index_values)
        > MAX_PER_PAGE_CLASS
    ):

        index_values = rng.choice(
            index_values,
            size=MAX_PER_PAGE_CLASS,
            replace=False,
        )

    sampled_indices.append(
        index_values
    )


require(
    sampled_indices,
    "Stable training sample is empty.",
)


sampled_indices = np.concatenate(
    sampled_indices
)


stable = (
    eligible.loc[
        sampled_indices
    ]
    .sort_values(
        [
            "source_group",
            "page",
            "y",
            "x",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


stable[
    "sample_id"
] = np.arange(
    len(stable),
    dtype=np.int64,
)


require(
    stable[
        "label"
    ].nunique()
    == 2,
    (
        "Stable training sample does not contain "
        "both PRINT and ANNOTATION."
    ),
)


del supervised
del sampled_indices

gc.collect()


# ------------------------------------------------------------
# Source-group OOF folds
# ------------------------------------------------------------

groups = stable[
    "source_group"
].astype(str)


unique_groups = groups.nunique()


require(
    unique_groups >= 3,
    (
        "At least three source groups/page groups "
        "are required."
    ),
)


splitter = (
    LeaveOneGroupOut()

    if unique_groups <= 10

    else GroupKFold(
        n_splits=min(
            10,
            unique_groups,
        )
    )
)


stable[
    "fold"
] = np.full(
    len(stable),
    -1,
    dtype=np.int16,
)


group_to_fold = {}

split_rows = []


for fold_id, (
    train_idx,
    test_idx,
) in enumerate(
    splitter.split(
        stable,
        stable[
            "label"
        ],
        groups,
    ),
    start=1,
):

    train_groups = set(
        groups.iloc[
            train_idx
        ]
    )

    test_groups = set(
        groups.iloc[
            test_idx
        ]
    )

    overlap = (
        train_groups
        & test_groups
    )


    require(
        not overlap,
        (
            f"Fold {fold_id}: source-group "
            f"leakage {sorted(overlap)}"
        ),
    )


    stable.loc[
        test_idx,
        "fold",
    ] = fold_id


    for group_name in test_groups:

        require(
            group_name
            not in group_to_fold,
            (
                "Source group assigned twice: "
                f"{group_name}"
            ),
        )

        group_to_fold[
            group_name
        ] = fold_id


    split_rows.append({
        "fold":
            fold_id,

        "train_groups":
            len(
                train_groups
            ),

        "test_groups":
            len(
                test_groups
            ),

        "test_pages":
            stable.iloc[
                test_idx
            ].page.nunique(),

        "test_rows":
            len(
                test_idx
            ),

        "test_source_groups":
            "|".join(
                sorted(
                    test_groups
                )
            ),

        "overlap_groups":
            0,
    })


# Source groups represented only by UNCERTAIN rows receive an
# evaluation fold but contribute no supervised training rows.
for group_name in eligible[
    "source_group"
].unique():

    if (
        group_name
        not in group_to_fold
    ):

        group_to_fold[
            group_name
        ] = 1


eligible[
    "fold"
] = eligible[
    "source_group"
].map(
    group_to_fold
)


require(
    (
        stable[
            "fold"
        ]
        > 0
    ).all(),
    "Incomplete stable fold assignment.",
)


require(
    eligible[
        "fold"
    ].notna().all(),
    "Incomplete eligible fold assignment.",
)


eligible[
    "fold"
] = eligible[
    "fold"
].astype(
    np.int16
)


stable[
    "fold"
] = stable[
    "fold"
].astype(
    np.int16
)


# ------------------------------------------------------------
# Durable rosters
# ------------------------------------------------------------

stable.to_parquet(
    TABLES
    / "stable_training_sample.parquet",
    index=False,
)


eligible.to_parquet(
    TABLES
    / "eligible_pixel_manifest.parquet",
    index=False,
)


pd.DataFrame(
    split_rows
).to_csv(
    AUDIT
    / "source_group_split_audit.csv",
    index=False,
)


pd.DataFrame([
    {
        "source_group":
            group_name,

        "fold":
            fold_id,
    }

    for group_name, fold_id
    in sorted(
        group_to_fold.items()
    )
]).to_csv(
    INPUTS
    / "fold_map.csv",
    index=False,
)


gc.collect()


print(
    "Stable supervised rows:",
    len(stable),
)

print(
    stable[
        "label"
    ].value_counts().rename(
        index={
            LABEL_PRINT:
                "PRINT_KEEP",

            LABEL_ANNOTATION:
                "ANNOTATION_CLEAN",
        }
    )
)

print(
    "Complete eligible rows:",
    len(eligible),
)

print(
    "Folds:",
    sorted(
        stable[
            "fold"
        ].unique()
    ),
)

# 6. Fast bounded-memory tensor builder and compact CNN

PERFORMANCE-ONLY REPLACEMENT
----------------------------
This cell does not change:
  - the eligible or stable populations;
  - page/source-group folds;
  - channel names, order or normalisation;
  - 33 x 33 patches or reflect padding;
  - model architecture;
  - batch size, epochs, optimiser, loss or random seeds;
  - calibration or decision rules.

It changes only the storage/extraction implementation:
  1. immutable cached .npy channels are staged from Drive to
     the Colab local SSD;
  2. local files are checked against the Drive cache hashes;
  3. reflected row/column indices are calculated once per
     sample rather than once per channel;
  4. patches are read from local read-only memmaps;
  5. DataLoader workers remain zero for deterministic,
     bounded-memory operation.

In [ ]:
# @title
import gc
import json
import os
import random
import shutil
import time

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Fixed performance configuration
# ============================================================

LOCAL_CHANNEL_CACHE = Path(
    "/content/n88_observable_channel_cache"
)

LOCAL_CHANNEL_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

# Recopy a page only when its local manifest/files are absent or
# do not match the durable Drive cache.
FORCE_LOCAL_CACHE_REFRESH = False

# Zero workers avoids multiprocessing duplication of Dataset and
# memmap state. Local SSD removes the original Drive-I/O bottleneck.
DATALOADER_WORKERS = 0

# Keep the experiment's existing batch size. This cell does not
# alter BATCH, EPOCHS, LEARNING_RATE, WEIGHT_DECAY or SEED.
require(
    "BATCH" in globals(),
    "BATCH must be defined before Cell 6.",
)

require(
    "EPOCHS" in globals(),
    "EPOCHS must be defined before Cell 6.",
)

require(
    "LEARNING_RATE" in globals(),
    "LEARNING_RATE must be defined before Cell 6.",
)

require(
    "WEIGHT_DECAY" in globals(),
    "WEIGHT_DECAY must be defined before Cell 6.",
)

require(
    "SEED" in globals(),
    "SEED must be defined before Cell 6.",
)

require(
    "PATCH" in globals(),
    "PATCH must be defined before Cell 6.",
)

require(
    "CHANNELS" in globals(),
    "CHANNELS must be defined before Cell 6.",
)

require(
    PATCH % 2 == 1,
    "PATCH must be odd.",
)

HALF = PATCH // 2


# ============================================================
# Schema checks
# ============================================================

require(
    "CHANNEL_CACHE" in globals(),
    "Run Cell 4 before this cell: CHANNEL_CACHE is missing.",
)

require(
    "CACHED_CHANNEL_DTYPES" in globals(),
    "Run Cell 4 before this cell: cache schema is missing.",
)

require(
    "runtime" in globals(),
    "Run Cell 4 before this cell: runtime is missing.",
)

require(
    "stable" in globals(),
    "Run Cell 5 before this cell: stable roster is missing.",
)

require(
    "eligible" in globals(),
    "Run Cell 5 before this cell: eligible roster is missing.",
)

require(
    set(CHANNELS) <= set(CACHED_CHANNEL_DTYPES),
    (
        "Tensor channels absent from the cache schema: "
        f"{sorted(set(CHANNELS) - set(CACHED_CHANNEL_DTYPES))}"
    ),
)

require(
    len(CHANNELS) > 0,
    "CHANNELS is empty.",
)

require(
    not stable.empty,
    "Stable supervised roster is empty.",
)

require(
    not eligible.empty,
    "Eligible roster is empty.",
)

for required_column in [
    "page",
    "y",
    "x",
    "label",
]:
    require(
        required_column in stable.columns,
        f"Stable roster lacks {required_column!r}.",
    )

for required_column in [
    "page",
    "y",
    "x",
    "label",
]:
    require(
        required_column in eligible.columns,
        f"Eligible roster lacks {required_column!r}.",
    )


# ============================================================
# Local SSD cache helpers
# ============================================================

def local_page_dir(page):
    return LOCAL_CHANNEL_CACHE / str(page)


def local_channel_path(page, channel):
    return local_page_dir(page) / f"{channel}.npy"


def local_manifest_path(page):
    return local_page_dir(page) / "cache_manifest.json"


def files_equal_by_sha256(source, target):
    return (
        Path(target).exists()
        and Path(source).stat().st_size == Path(target).stat().st_size
        and sha256_file(source) == sha256_file(target)
    )


def local_page_is_valid(page):
    source_manifest = page_cache_manifest_path(page)
    target_manifest = local_manifest_path(page)

    if FORCE_LOCAL_CACHE_REFRESH:
        return False

    if not source_manifest.exists() or not target_manifest.exists():
        return False

    try:
        source_record = json.loads(source_manifest.read_text())
        target_record = json.loads(target_manifest.read_text())

        if source_record != target_record:
            return False

        expected_shape = tuple(source_record["shape"])

        for channel, expected_dtype in CACHED_CHANNEL_DTYPES.items():
            source = channel_path(page, channel)
            target = local_channel_path(page, channel)

            if not source.exists() or not target.exists():
                return False

            source_channel_record = source_record["channels"][channel]

            if target.stat().st_size != int(source_channel_record["bytes"]):
                return False

            if sha256_file(target) != source_channel_record["sha256"]:
                return False

            array = np.load(
                target,
                mmap_mode="r",
                allow_pickle=False,
            )

            valid = (
                array.ndim == 2
                and array.shape == expected_shape
                and array.dtype == np.dtype(expected_dtype)
            )

            del array

            if not valid:
                return False

        return True

    except Exception:
        return False


def stage_page_to_local_ssd(page):
    page = str(page)
    source_dir = Path(CHANNEL_CACHE) / page
    target_dir = local_page_dir(page)

    require(
        source_dir.exists(),
        f"{page}: durable channel cache is missing: {source_dir}",
    )

    if local_page_is_valid(page):
        return "REUSED"

    temporary_dir = LOCAL_CHANNEL_CACHE / f".{page}.copying"

    if temporary_dir.exists():
        shutil.rmtree(temporary_dir)

    temporary_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    try:
        source_manifest_path = page_cache_manifest_path(page)
        source_record = json.loads(
            source_manifest_path.read_text()
        )

        for channel in CACHED_CHANNEL_DTYPES:
            source = channel_path(page, channel)
            target = temporary_dir / f"{channel}.npy"

            require(
                source.exists(),
                f"{page}: durable cached channel missing: {source}",
            )

            shutil.copy2(source, target)

            expected = source_record["channels"][channel]

            require(
                target.stat().st_size == int(expected["bytes"]),
                f"{page}/{channel}: local copy byte-size mismatch.",
            )

            require(
                sha256_file(target) == expected["sha256"],
                f"{page}/{channel}: local copy SHA-256 mismatch.",
            )

        shutil.copy2(
            source_manifest_path,
            temporary_dir / "cache_manifest.json",
        )

        if target_dir.exists():
            shutil.rmtree(target_dir)

        temporary_dir.replace(target_dir)

        require(
            local_page_is_valid(page),
            f"{page}: local SSD cache failed post-copy validation.",
        )

        return "COPIED"

    except Exception:
        if temporary_dir.exists():
            shutil.rmtree(temporary_dir)
        raise


# ============================================================
# Release Drive-backed memmaps before local staging
# ============================================================

try:
    runtime.clear()
except Exception:
    pass

runtime = {}

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# Stage immutable channel cache to local SSD
# ============================================================

page_list = sorted(
    set(stable.page.astype(str))
    | set(eligible.page.astype(str))
)

require(
    len(page_list) > 0,
    "Tensor page roster is empty.",
)

local_stage_rows = []

stage_started = time.time()

for page_number, page in enumerate(
    page_list,
    start=1,
):
    page_started = time.time()

    status = stage_page_to_local_ssd(page)

    page_bytes = sum(
        local_channel_path(
            page,
            channel,
        ).stat().st_size
        for channel in CACHED_CHANNEL_DTYPES
    )

    local_stage_rows.append({
        "page": page,
        "status": status,
        "bytes": int(page_bytes),
        "mib": float(page_bytes / 1024**2),
        "seconds": float(time.time() - page_started),
    })

    print(
        f"[{page_number:03d}/{len(page_list):03d}] "
        f"{page}: local cache {status} | "
        f"{page_bytes / 1024**2:.1f} MiB"
    )

local_stage_audit = pd.DataFrame(local_stage_rows)

local_stage_audit.to_csv(
    AUDIT / "local_ssd_channel_cache_audit.csv",
    index=False,
)


# ============================================================
# Open local read-only memmaps
# ============================================================

class LocalDiskBackedPage(dict):
    def __init__(self, page):
        super().__init__()

        self.page = str(page)

        for channel, expected_dtype in CACHED_CHANNEL_DTYPES.items():
            path = local_channel_path(
                self.page,
                channel,
            )

            array = np.load(
                path,
                mmap_mode="r",
                allow_pickle=False,
            )

            require(
                array.ndim == 2,
                f"{self.page}/{channel}: local cached channel is not 2D.",
            )

            require(
                array.dtype == np.dtype(expected_dtype),
                (
                    f"{self.page}/{channel}: dtype {array.dtype} "
                    f"!= {np.dtype(expected_dtype)}"
                ),
            )

            self[channel] = array


for page in page_list:
    runtime[page] = LocalDiskBackedPage(page)

    shapes = {
        tuple(array.shape)
        for array in runtime[page].values()
    }

    require(
        len(shapes) == 1,
        f"{page}: local cached channels have inconsistent shapes: {shapes}",
    )


# ============================================================
# Compact coordinate encoding
# ============================================================

page_index = {
    page: index
    for index, page in enumerate(page_list)
}


def encode(frame):
    pages_encoded = frame.page.astype(str).map(
        page_index
    )

    require(
        pages_encoded.notna().all(),
        "Roster contains a page absent from page_index.",
    )

    return (
        pages_encoded.to_numpy(
            dtype=np.int16,
            copy=True,
        ),
        frame.y.to_numpy(
            dtype=np.int32,
            copy=True,
        ),
        frame.x.to_numpy(
            dtype=np.int32,
            copy=True,
        ),
        frame.label.to_numpy(
            dtype=np.int8,
            copy=True,
        ),
    )


(
    stable_pid,
    stable_y,
    stable_x,
    stable_truth,
) = encode(stable)

(
    all_pid,
    all_y,
    all_x,
    all_truth,
) = encode(eligible)


# ============================================================
# Exact reflect-padding coordinate mapping
#
# np.pad(..., mode="reflect") semantics are reproduced without
# materialising padded full-page float32 arrays.
# ============================================================

_OFFSET = np.arange(
    -HALF,
    HALF + 1,
    dtype=np.int64,
)


def reflected_indices(center, length):
    require(
        length > 1,
        "Reflect padding requires an axis length greater than one.",
    )

    values = int(center) + _OFFSET
    period = 2 * int(length) - 2
    values = np.mod(values, period)

    return np.where(
        values < length,
        values,
        period - values,
    ).astype(
        np.intp,
        copy=False,
    )


_BINARY_CHANNELS = {
    "reviewed_mask",
    "ambiguous",
    "p_print_available",
    "edge",
    "horizontal_support",
    "vertical_support",
    "compact_support",
}

_PROBABILITY_CHANNELS = {
    "p_annotation",
    "p_print_value",
}

_UINT8_INTENSITY_CHANNELS = {
    "observed",
    "local_normalised",
}


require(
    _BINARY_CHANNELS <= set(CHANNELS),
    (
        "Expected binary tensor channels absent from CHANNELS: "
        f"{sorted(_BINARY_CHANNELS - set(CHANNELS))}"
    ),
)

require(
    _PROBABILITY_CHANNELS <= set(CHANNELS),
    (
        "Expected probability tensor channels absent from CHANNELS: "
        f"{sorted(_PROBABILITY_CHANNELS - set(CHANNELS))}"
    ),
)


# ============================================================
# Fast patch extraction
# ============================================================

def get_patch(pid, y, x):
    page = page_list[int(pid)]
    page_channels = runtime[page]

    height, width = page_channels[
        CHANNELS[0]
    ].shape

    require(
        0 <= int(y) < height
        and 0 <= int(x) < width,
        f"{page}/{y}/{x}: patch centre is outside the page.",
    )

    # Calculated once per sample, not once per channel.
    y_indices = reflected_indices(
        y,
        height,
    )

    x_indices = reflected_indices(
        x,
        width,
    )

    patch = np.empty(
        (
            len(CHANNELS),
            PATCH,
            PATCH,
        ),
        dtype=np.float32,
    )

    selector = np.ix_(
        y_indices,
        x_indices,
    )

    for channel_index, key in enumerate(CHANNELS):
        source = page_channels[key]

        # Advanced indexing materialises only this 33 x 33 patch.
        channel_patch = np.asarray(
            source[selector]
        )

        target = patch[channel_index]

        if key in _UINT8_INTENSITY_CHANNELS:
            np.multiply(
                channel_patch,
                np.float32(1.0 / 255.0),
                out=target,
                casting="unsafe",
            )

        elif key == "mask_depth":
            np.divide(
                channel_patch,
                np.float32(HALF),
                out=target,
                casting="unsafe",
            )

            np.clip(
                target,
                0.0,
                1.0,
                out=target,
            )

        elif (
            key in _BINARY_CHANNELS
            or key in _PROBABILITY_CHANNELS
        ):
            np.copyto(
                target,
                channel_patch,
                casting="unsafe",
            )

        else:
            raise RuntimeError(
                f"Unhandled tensor channel: {key}"
            )

    if not np.isfinite(patch).all():
        bad_channels = []

        for channel_index, key in enumerate(CHANNELS):
            channel_patch = patch[channel_index]
            bad_count = int(
                (~np.isfinite(channel_patch)).sum()
            )

            if bad_count:
                bad_channels.append({
                    "channel": key,
                    "nonfinite": bad_count,
                    "nan": int(np.isnan(channel_patch).sum()),
                    "positive_inf": int(np.isposinf(channel_patch).sum()),
                    "negative_inf": int(np.isneginf(channel_patch).sum()),
                })

        raise RuntimeError(
            f"{page}/{y}/{x}: tensor contains "
            f"non-finite values: {bad_channels}"
        )

    return patch


# ============================================================
# Dataset and unchanged compact CNN
# ============================================================

class ArrayPatchDataset(Dataset):
    def __init__(
        self,
        idx,
        pids,
        ys,
        xs,
        label,
    ):
        self.idx = np.asarray(
            idx,
            dtype=np.int64,
        )

        self.pids = pids
        self.ys = ys
        self.xs = xs
        self.label = label

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = int(self.idx[i])

        patch = get_patch(
            self.pids[j],
            self.ys[j],
            self.xs[j],
        )

        return (
            torch.from_numpy(patch),
            torch.tensor(
                int(self.label[j]),
                dtype=torch.long,
            ),
        )


class TopologyCNN(nn.Module):
    def __init__(
        self,
        nc=len(CHANNELS),
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(
                nc,
                24,
                3,
                padding=1,
            ),
            nn.ReLU(),
            nn.BatchNorm2d(24),
            nn.MaxPool2d(2),
            nn.Conv2d(
                24,
                48,
                3,
                padding=1,
            ),
            nn.ReLU(),
            nn.BatchNorm2d(48),
            nn.MaxPool2d(2),
            nn.Conv2d(
                48,
                64,
                3,
                padding=1,
            ),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.head = nn.Linear(
            64,
            2,
        )

    def forward(self, x):
        return self.head(
            self.net(x).flatten(1)
        )


# ============================================================
# Deterministic DataLoader
# ============================================================

def worker_init(worker_id):
    worker_seed = SEED + int(worker_id)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def loader(
    idx,
    pids,
    ys,
    xs,
    label,
    shuffle,
    seed,
):
    generator = torch.Generator()
    generator.manual_seed(int(seed))

    return DataLoader(
        ArrayPatchDataset(
            idx,
            pids,
            ys,
            xs,
            label,
        ),
        batch_size=BATCH,
        shuffle=bool(shuffle),
        num_workers=DATALOADER_WORKERS,
        pin_memory=True,
        generator=generator,
        worker_init_fn=(
            worker_init
            if DATALOADER_WORKERS > 0
            else None
        ),
        persistent_workers=False,
        drop_last=False,
    )


# ============================================================
# Unchanged fold fitting and prediction functions
# ============================================================

def fit_fold(train_idx):
    reset_seed()

    model = TopologyCNN().to(DEVICE)

    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    counts = np.bincount(
        stable_truth[train_idx],
        minlength=2,
    )

    weights = torch.tensor(
        counts.sum()
        / np.maximum(
            counts,
            1,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    loss_function = nn.CrossEntropyLoss(
        weight=weights
    )

    history = []

    for epoch in range(EPOCHS):
        epoch_started = time.time()
        total_loss = 0.0
        row_count = 0

        model.train()

        training_loader = loader(
            train_idx,
            stable_pid,
            stable_y,
            stable_x,
            stable_truth,
            True,
            SEED,
        )

        for xb, yb in training_loader:
            xb = xb.to(
                DEVICE,
                non_blocking=True,
            )

            yb = yb.to(
                DEVICE,
                non_blocking=True,
            )

            optimiser.zero_grad(
                set_to_none=True
            )

            loss = loss_function(
                model(xb),
                yb,
            )

            loss.backward()
            optimiser.step()

            batch_rows = len(yb)

            total_loss += (
                loss.detach().item()
                * batch_rows
            )

            row_count += batch_rows

        mean_loss = total_loss / max(
            row_count,
            1,
        )

        history.append(mean_loss)

        print(
            f"  epoch {epoch + 1:02d}/{EPOCHS}: "
            f"loss={mean_loss:.6f} | "
            f"seconds={time.time() - epoch_started:.1f}"
        )

        del training_loader
        gc.collect()

    return model, history


@torch.inference_mode()
def predict(
    model,
    idx,
    pids,
    ys,
    xs,
    label,
):
    output = []

    model.eval()

    prediction_loader = loader(
        idx,
        pids,
        ys,
        xs,
        label,
        False,
        SEED,
    )

    for xb, _ in prediction_loader:
        probabilities = torch.softmax(
            model(
                xb.to(
                    DEVICE,
                    non_blocking=True,
                )
            ),
            dim=1,
        )[:, 1]

        output.append(
            probabilities.cpu().numpy()
        )

    del prediction_loader

    if output:
        return np.concatenate(
            output
        ).astype(
            np.float64,
            copy=False,
        )

    return np.empty(
        0,
        dtype=np.float64,
    )


# ============================================================
# Tensor-builder smoke test
#
# Use the COMPLETE eligible roster, not the supervised stable
# sample.
#
# Some tensor pages can legitimately contain no PRINT/ANNOTATION
# row selected into `stable`, while still containing eligible
# AMBIGUOUS rows required for evaluation.
# ============================================================

smoke_rows = []


eligible_page_set = set(
    eligible[
        "page"
    ].astype(str)
)


tensor_page_set = set(
    page_list
)


# Every tensor page participating in Notebook 88 must have at
# least one eligible coordinate from which a patch can be tested.
missing_eligible_pages = sorted(
    tensor_page_set
    - eligible_page_set
)


require(
    not missing_eligible_pages,
    (
        "Tensor pages lack an eligible row for smoke testing: "
        f"{missing_eligible_pages}"
    ),
)


# Select exactly one deterministic eligible coordinate per page.
#
# Sorting means this is invariant to the current DataFrame row
# ordering.
smoke = (
    eligible[
        eligible[
            "page"
        ].astype(str).isin(
            page_list
        )
    ]
    .sort_values(
        [
            "page",
            "y",
            "x",
        ],
        kind="mergesort",
    )
    .groupby(
        "page",
        sort=False,
        as_index=False,
    )
    .first()
)


require(
    smoke[
        "page"
    ].nunique()
    == len(
        page_list
    ),
    (
        "Smoke-test roster does not contain exactly one "
        "eligible row for every tensor page."
    ),
)


for smoke_number, row in enumerate(
    smoke.itertuples(
        index=False
    ),
    start=1,
):

    page = str(
        row.page
    )


    require(
        page in page_index,
        (
            f"{page}: smoke-test page is absent "
            "from the tensor page index."
        ),
    )


    pid = page_index[
        page
    ]


    patch = get_patch(
        pid,
        int(
            row.y
        ),
        int(
            row.x
        ),
    )


    require(
        patch.shape
        == (
            len(
                CHANNELS
            ),
            PATCH,
            PATCH,
        ),
        (
            f"{page}: smoke-test patch shape "
            f"{patch.shape} != "
            f"{(len(CHANNELS), PATCH, PATCH)}"
        ),
    )


    require(
        patch.dtype
        == np.float32,
        (
            f"{page}: smoke-test patch dtype "
            f"{patch.dtype} != float32"
        ),
    )


    require(
        np.isfinite(
            patch
        ).all(),
        (
            f"{page}: smoke-test tensor "
            "contains non-finite values."
        ),
    )


    smoke_rows.append({
        "page":
            page,

        "y":
            int(
                row.y
            ),

        "x":
            int(
                row.x
            ),

        "channels":
            int(
                patch.shape[
                    0
                ]
            ),

        "patch":
            int(
                patch.shape[
                    1
                ]
            ),

        "finite":
            True,
    })


    del patch


    if (
        smoke_number % 10
        == 0
        or smoke_number
        == len(
            smoke
        )
    ):

        print(
            f"Smoke tested "
            f"{smoke_number}/"
            f"{len(smoke)} pages"
        )


    gc.collect()


smoke_audit = pd.DataFrame(
    smoke_rows
)


smoke_audit.to_csv(
    AUDIT
    / "tensor_builder_smoke_test.csv",
    index=False,
)


require(
    len(
        smoke_audit
    )
    == len(
        page_list
    ),
    (
        "Tensor smoke-test audit does not cover "
        "the complete tensor page roster."
    ),
)


print()
print(
    "Tensor smoke test: PASS"
)

print(
    "Pages tested:",
    len(
        smoke_audit
    ),
)

print(
    "Channels:",
    len(
        CHANNELS
    ),
)

print(
    "Patch:",
    PATCH,
    "x",
    PATCH,
)


gc.collect()

print()
print("=" * 72)
print("FAST BOUNDED-MEMORY TENSOR BUILDER READY")
print("=" * 72)
print(
    "Patch:",
    f"{PATCH} x {PATCH}",
)
print(
    "Channels:",
    len(CHANNELS),
)
print(
    "DataLoader workers:",
    DATALOADER_WORKERS,
)
print(
    "Training batch size:",
    BATCH,
)
print(
    "Local cache root:",
    LOCAL_CHANNEL_CACHE,
)
print(
    "Local cache size:",
    f"{local_stage_audit.bytes.sum() / 1024**3:.2f} GiB",
)
print(
    "Local staging seconds:",
    f"{time.time() - stage_started:.1f}",
)
print(
    "Scientific contract changed:",
    False,
)


# 6A. Bounded-memory preflight audit of every cached A5 channel

PURPOSE
-------
Prove BEFORE model training that:

  1. every page required by the eligible/stable rosters exists
     in the disk-backed runtime;
  2. every declared channel exists;
  3. every channel has the expected page shape;
  4. every channel has the expected dtype;
  5. every channel contains zero NaN / +Inf / -Inf values;
  6. probability channels remain inside [0, 1];
  7. binary channels remain binary;
  8. every supervised/eligible coordinate is inside the page;
  9. every eligible coordinate is AMBIGUOUS;
 10. a representative patch can be constructed for every page.

MEMORY CONTRACT
---------------
Do NOT call np.isfinite() over an entire full-resolution page
in one operation.

Audit in row blocks so temporary RAM remains bounded.

This cell does not alter any input or cached channel.

In [ ]:
# @title
'''
# ============================================================
# 6A. Bounded-memory preflight audit of every cached A5 channel
#
# PURPOSE
# -------
# Prove BEFORE model training that:
#
#   1. every page required by the eligible/stable rosters exists
#      in the disk-backed runtime;
#   2. every declared channel exists;
#   3. every channel has the expected page shape;
#   4. every channel has the expected dtype;
#   5. every channel contains zero NaN / +Inf / -Inf values;
#   6. probability channels remain inside [0, 1];
#   7. binary channels remain binary;
#   8. every supervised/eligible coordinate is inside the page;
#   9. every eligible coordinate is AMBIGUOUS;
#  10. a representative patch can be constructed for every page.
#
# MEMORY CONTRACT
# ---------------
# Do NOT call np.isfinite() over an entire full-resolution page
# in one operation.
#
# Audit in row blocks so temporary RAM remains bounded.
#
# This cell does not alter any input or cached channel.
# ============================================================

import gc
import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

# Number of image rows to inspect per block.
#
# 256 rows is deliberately conservative for Colab RAM.
PREFLIGHT_ROW_BLOCK = 256


# Number of representative patch checks per page.
#
# First, middle and last eligible rows are sufficient to verify
# patch construction across the page without materialising the
# complete tensor population.
PREFLIGHT_PATCHES_PER_PAGE = 3


# Channels whose cached representation must be binary 0/1.
PREFLIGHT_BINARY_CHANNELS = {
    "reviewed_mask",
    "ambiguous",
    "p_print_available",
    "edge",
    "horizontal_support",
    "vertical_support",
    "compact_support",
}


# Probability channels must be finite and inside [0,1].
PREFLIGHT_PROBABILITY_CHANNELS = {
    "p_annotation",
    "p_print_value",
}


# Expected cached dtypes from memory-safe Cell 4.
#
# Use Cell 4's declared schema where available rather than
# silently inventing another representation here.
if "CACHED_CHANNEL_DTYPES" not in globals():

    CACHED_CHANNEL_DTYPES = {
        "observed":
            np.uint8,

        "a1":
            np.uint8,

        "paper":
            np.uint8,

        "reviewed_mask":
            np.uint8,

        "ambiguous":
            np.uint8,

        "local_normalised":
            np.uint8,

        "mask_depth":
            np.float32,

        "p_annotation":
            np.float32,

        "p_print":
            np.float32,

        "edge":
            np.uint8,

        "horizontal_support":
            np.uint8,

        "vertical_support":
            np.uint8,

        "compact_support":
            np.uint8,
    }


# ============================================================
# Structural prerequisites
# ============================================================

require(
    "runtime" in globals(),
    (
        "runtime does not exist. "
        "Run memory-safe Cell 4 first."
    ),
)


require(
    "page_list" in globals(),
    (
        "page_list does not exist. "
        "Run Cell 6 before this preflight cell."
    ),
)


require(
    "get_patch" in globals(),
    (
        "get_patch() does not exist. "
        "Run Cell 6 before this preflight cell."
    ),
)


require(
    "eligible" in globals(),
    (
        "eligible roster does not exist. "
        "Run Cell 5 first."
    ),
)


require(
    "stable" in globals(),
    (
        "stable roster does not exist. "
        "Run Cell 5 first."
    ),
)


require(
    len(page_list) > 0,
    "Preflight page roster is empty.",
)


# ============================================================
# Utility: bounded channel audit
# ============================================================

def audit_channel_bounded(
    page,
    channel,
    array,
):

    """
    Audit one disk-backed page/channel in bounded row blocks.

    No whole-page Boolean finite array is created.
    """

    height, width = array.shape

    nonfinite_count = 0
    nan_count = 0
    positive_inf_count = 0
    negative_inf_count = 0

    below_zero_count = 0
    above_one_count = 0

    nonbinary_count = 0

    minimum = None
    maximum = None


    for y0 in range(
        0,
        height,
        PREFLIGHT_ROW_BLOCK,
    ):

        y1 = min(
            height,
            y0
            + PREFLIGHT_ROW_BLOCK,
        )


        # np.asarray() materialises only this row block.
        block = np.asarray(
            array[
                y0:y1,
                :
            ]
        )


        # -----------------------------------------------
        # Finite audit
        # -----------------------------------------------

        if np.issubdtype(
            block.dtype,
            np.floating,
        ):

            block_nan = int(
                np.isnan(
                    block
                ).sum()
            )

            block_posinf = int(
                np.isposinf(
                    block
                ).sum()
            )

            block_neginf = int(
                np.isneginf(
                    block
                ).sum()
            )


            nan_count += (
                block_nan
            )

            positive_inf_count += (
                block_posinf
            )

            negative_inf_count += (
                block_neginf
            )

            nonfinite_count += (
                block_nan
                + block_posinf
                + block_neginf
            )


            finite_block = block[
                np.isfinite(
                    block
                )
            ]


        else:

            # Integer data cannot contain NaN or Inf.
            finite_block = block


        # -----------------------------------------------
        # Min/max without trusting non-finite values
        # -----------------------------------------------

        if finite_block.size:

            block_minimum = float(
                finite_block.min()
            )

            block_maximum = float(
                finite_block.max()
            )


            minimum = (
                block_minimum

                if minimum is None

                else min(
                    minimum,
                    block_minimum,
                )
            )


            maximum = (
                block_maximum

                if maximum is None

                else max(
                    maximum,
                    block_maximum,
                )
            )


        # -----------------------------------------------
        # Probability range
        # -----------------------------------------------

        if (
            channel
            in PREFLIGHT_PROBABILITY_CHANNELS
        ):

            # Count only finite values for range failures.
            if np.issubdtype(
                block.dtype,
                np.floating,
            ):

                finite_mask = np.isfinite(
                    block
                )

                below_zero_count += int(
                    (
                        finite_mask
                        & (
                            block < 0.0
                        )
                    ).sum()
                )

                above_one_count += int(
                    (
                        finite_mask
                        & (
                            block > 1.0
                        )
                    ).sum()
                )

                del finite_mask

            else:

                below_zero_count += int(
                    (
                        block < 0
                    ).sum()
                )

                above_one_count += int(
                    (
                        block > 1
                    ).sum()
                )


        # -----------------------------------------------
        # Binary channel integrity
        # -----------------------------------------------

        if (
            channel
            in PREFLIGHT_BINARY_CHANNELS
        ):

            nonbinary_count += int(
                (
                    (
                        block != 0
                    )
                    & (
                        block != 1
                    )
                ).sum()
            )


        del finite_block
        del block

        gc.collect()


    return {
        "page":
            page,

        "channel":
            channel,

        "height":
            int(
                height
            ),

        "width":
            int(
                width
            ),

        "pixels":
            int(
                height
                * width
            ),

        "dtype":
            str(
                array.dtype
            ),

        "minimum":
            minimum,

        "maximum":
            maximum,

        "nonfinite":
            int(
                nonfinite_count
            ),

        "nan":
            int(
                nan_count
            ),

        "positive_inf":
            int(
                positive_inf_count
            ),

        "negative_inf":
            int(
                negative_inf_count
            ),

        "below_zero":
            int(
                below_zero_count
            ),

        "above_one":
            int(
                above_one_count
            ),

        "nonbinary":
            int(
                nonbinary_count
            ),
    }


# ============================================================
# Page roster integrity
# ============================================================

runtime_pages = set(
    str(
        page
    )
    for page
    in runtime
)


declared_pages = set(
    str(
        page
    )
    for page
    in page_list
)


eligible_pages = set(
    eligible[
        "page"
    ].astype(str)
)


stable_pages = set(
    stable[
        "page"
    ].astype(str)
)


require(
    eligible_pages
    <= runtime_pages,
    (
        "Eligible roster contains pages absent "
        "from runtime: "
        f"{sorted(eligible_pages - runtime_pages)}"
    ),
)


require(
    stable_pages
    <= runtime_pages,
    (
        "Stable roster contains pages absent "
        "from runtime: "
        f"{sorted(stable_pages - runtime_pages)}"
    ),
)


require(
    declared_pages
    == runtime_pages,
    (
        "page_list/runtime mismatch: "
        f"page_list only="
        f"{sorted(declared_pages - runtime_pages)}, "
        f"runtime only="
        f"{sorted(runtime_pages - declared_pages)}"
    ),
)


# ============================================================
# Run complete channel audit
# ============================================================

audit_rows = []

failure_rows = []

page_rows = []


for page_number, page in enumerate(
    page_list,
    start=1,
):

    page = str(
        page
    )


    page_runtime = runtime[
        page
    ]


    # --------------------------------------------------------
    # Every expected channel must exist.
    # --------------------------------------------------------

    missing_channels = [
        channel

        for channel
        in CACHED_CHANNEL_DTYPES

        if channel
        not in page_runtime
    ]


    if missing_channels:

        failure_rows.append({
            "page":
                page,

            "channel":
                "<schema>",

            "failure":
                "MISSING_CHANNEL",

            "detail":
                "|".join(
                    missing_channels
                ),
        })


        print(
            f"{page}: FAIL - missing channels "
            f"{missing_channels}"
        )

        continue


    # --------------------------------------------------------
    # Determine authoritative page shape from observed.
    # --------------------------------------------------------

    observed = page_runtime[
        "observed"
    ]


    require(
        observed.ndim == 2,
        (
            f"{page}: observed channel "
            "is not two-dimensional."
        ),
    )


    page_shape = (
        observed.shape
    )


    page_failure_count = 0


    # ========================================================
    # Audit every cached channel
    # ========================================================

    for channel, expected_dtype in (
        CACHED_CHANNEL_DTYPES.items()
    ):

        array = page_runtime[
            channel
        ]


        # -----------------------------------------------
        # Shape
        # -----------------------------------------------

        if (
            array.ndim != 2
            or array.shape
            != page_shape
        ):

            failure_rows.append({
                "page":
                    page,

                "channel":
                    channel,

                "failure":
                    "SHAPE_MISMATCH",

                "detail":
                    (
                        f"channel={array.shape}; "
                        f"observed={page_shape}"
                    ),
            })

            page_failure_count += 1

            continue


        # -----------------------------------------------
        # dtype
        # -----------------------------------------------

        expected_dtype = np.dtype(
            expected_dtype
        )


        if (
            array.dtype
            != expected_dtype
        ):

            failure_rows.append({
                "page":
                    page,

                "channel":
                    channel,

                "failure":
                    "DTYPE_MISMATCH",

                "detail":
                    (
                        f"actual={array.dtype}; "
                        f"expected={expected_dtype}"
                    ),
            })

            page_failure_count += 1

            continue


        # -----------------------------------------------
        # Bounded data audit
        # -----------------------------------------------

        channel_audit = (
            audit_channel_bounded(
                page,
                channel,
                array,
            )
        )


        audit_rows.append(
            channel_audit
        )


        # -----------------------------------------------
        # Non-finite
        # -----------------------------------------------

        if (
            channel_audit[
                "nonfinite"
            ]
            > 0
        ):

            failure_rows.append({
                "page":
                    page,

                "channel":
                    channel,

                "failure":
                    "NONFINITE_VALUES",

                "detail":
                    (
                        f"nonfinite="
                        f"{channel_audit['nonfinite']}; "
                        f"nan="
                        f"{channel_audit['nan']}; "
                        f"+inf="
                        f"{channel_audit['positive_inf']}; "
                        f"-inf="
                        f"{channel_audit['negative_inf']}"
                    ),
            })

            page_failure_count += 1


        # -----------------------------------------------
        # Probability range
        # -----------------------------------------------

        if (
            channel
            in PREFLIGHT_PROBABILITY_CHANNELS
            and (
                channel_audit[
                    "below_zero"
                ]
                > 0
                or channel_audit[
                    "above_one"
                ]
                > 0
            )
        ):

            failure_rows.append({
                "page":
                    page,

                "channel":
                    channel,

                "failure":
                    "PROBABILITY_OUTSIDE_0_1",

                "detail":
                    (
                        f"below_zero="
                        f"{channel_audit['below_zero']}; "
                        f"above_one="
                        f"{channel_audit['above_one']}"
                    ),
            })

            page_failure_count += 1


        # -----------------------------------------------
        # Binary integrity
        # -----------------------------------------------

        if (
            channel
            in PREFLIGHT_BINARY_CHANNELS
            and channel_audit[
                "nonbinary"
            ]
            > 0
        ):

            failure_rows.append({
                "page":
                    page,

                "channel":
                    channel,

                "failure":
                    "NONBINARY_VALUES",

                "detail":
                    (
                        f"nonbinary="
                        f"{channel_audit['nonbinary']}"
                    ),
            })

            page_failure_count += 1


    # ========================================================
    # Eligible coordinate integrity
    # ========================================================

    page_eligible = eligible[
        eligible[
            "page"
        ].astype(str)
        == page
    ]


    require(
        not page_eligible.empty,
        (
            f"{page}: no rows in eligible roster."
        ),
    )


    eligible_y_page = page_eligible[
        "y"
    ].to_numpy(
        dtype=np.int32,
        copy=False,
    )

    eligible_x_page = page_eligible[
        "x"
    ].to_numpy(
        dtype=np.int32,
        copy=False,
    )


    in_bounds = (
        (
            eligible_y_page
            >= 0
        )
        & (
            eligible_y_page
            < page_shape[
                0
            ]
        )
        & (
            eligible_x_page
            >= 0
        )
        & (
            eligible_x_page
            < page_shape[
                1
            ]
        )
    )


    if not in_bounds.all():

        bad_count = int(
            (
                ~in_bounds
            ).sum()
        )


        failure_rows.append({
            "page":
                page,

            "channel":
                "<coordinates>",

            "failure":
                "ELIGIBLE_COORDINATE_OUT_OF_BOUNDS",

            "detail":
                f"rows={bad_count}",
        })


        page_failure_count += 1


    else:

        ambiguous_at_eligible = (
            page_runtime[
                "ambiguous"
            ][
                eligible_y_page,
                eligible_x_page,
            ]
            > 0
        )


        if not ambiguous_at_eligible.all():

            bad_count = int(
                (
                    ~ambiguous_at_eligible
                ).sum()
            )


            failure_rows.append({
                "page":
                    page,

                "channel":
                    "ambiguous",

                "failure":
                    "ELIGIBLE_NOT_AMBIGUOUS",

                "detail":
                    f"rows={bad_count}",
            })


            page_failure_count += 1


        del ambiguous_at_eligible


    # ========================================================
    # Stable training-coordinate integrity
    # ========================================================

    page_stable = stable[
        stable[
            "page"
        ].astype(str)
        == page
    ]


    if not page_stable.empty:

        stable_y_page = page_stable[
            "y"
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )

        stable_x_page = page_stable[
            "x"
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )


        stable_in_bounds = (
            (
                stable_y_page
                >= 0
            )
            & (
                stable_y_page
                < page_shape[
                    0
                ]
            )
            & (
                stable_x_page
                >= 0
            )
            & (
                stable_x_page
                < page_shape[
                    1
                ]
            )
        )


        if not stable_in_bounds.all():

            bad_count = int(
                (
                    ~stable_in_bounds
                ).sum()
            )


            failure_rows.append({
                "page":
                    page,

                "channel":
                    "<coordinates>",

                "failure":
                    "STABLE_COORDINATE_OUT_OF_BOUNDS",

                "detail":
                    f"rows={bad_count}",
            })


            page_failure_count += 1


        del stable_y_page
        del stable_x_page
        del stable_in_bounds


    # ========================================================
    # Representative real tensor checks
    #
    # This invokes the exact get_patch() that Cell 7 will use.
    # ========================================================

    eligible_positions = np.linspace(
        0,
        len(
            page_eligible
        )
        - 1,
        num=min(
            PREFLIGHT_PATCHES_PER_PAGE,
            len(
                page_eligible
            ),
        ),
        dtype=int,
    )


    page_pid = page_index[
        page
    ]


    for position in np.unique(
        eligible_positions
    ):

        candidate = page_eligible.iloc[
            int(
                position
            )
        ]


        patch_y = int(
            candidate.y
        )

        patch_x = int(
            candidate.x
        )


        try:

            test_patch = get_patch(
                page_pid,
                patch_y,
                patch_x,
            )


            require(
                test_patch.shape
                == (
                    len(
                        CHANNELS
                    ),
                    PATCH,
                    PATCH,
                ),
                (
                    f"{page}/{patch_y}/{patch_x}: "
                    f"patch shape {test_patch.shape} "
                    "is incorrect."
                ),
            )


            require(
                test_patch.dtype
                == np.float32,
                (
                    f"{page}/{patch_y}/{patch_x}: "
                    f"patch dtype {test_patch.dtype} "
                    "is not float32."
                ),
            )


            if not np.isfinite(
                test_patch
            ).all():

                bad_channels = []


                for channel_index, channel in enumerate(
                    CHANNELS
                ):

                    channel_patch = test_patch[
                        channel_index
                    ]


                    bad_count = int(
                        (
                            ~np.isfinite(
                                channel_patch
                            )
                        ).sum()
                    )


                    if bad_count:

                        bad_channels.append(
                            {
                                "channel":
                                    channel,

                                "nonfinite":
                                    bad_count,

                                "nan":
                                    int(
                                        np.isnan(
                                            channel_patch
                                        ).sum()
                                    ),

                                "positive_inf":
                                    int(
                                        np.isposinf(
                                            channel_patch
                                        ).sum()
                                    ),

                                "negative_inf":
                                    int(
                                        np.isneginf(
                                            channel_patch
                                        ).sum()
                                    ),
                            }
                        )


                raise RuntimeError(
                    (
                        f"{page}/{patch_y}/{patch_x}: "
                        "representative tensor contains "
                        f"non-finite values: {bad_channels}"
                    )
                )


            del test_patch


        except Exception as error:

            failure_rows.append({
                "page":
                    page,

                "channel":
                    "<tensor>",

                "failure":
                    "PATCH_BUILD_FAILURE",

                "detail":
                    (
                        f"y={patch_y}; "
                        f"x={patch_x}; "
                        f"error={error}"
                    ),
            })

            page_failure_count += 1


    # ========================================================
    # Page summary
    # ========================================================

    page_rows.append({
        "page":
            page,

        "height":
            int(
                page_shape[
                    0
                ]
            ),

        "width":
            int(
                page_shape[
                    1
                ]
            ),

        "eligible_rows":
            int(
                len(
                    page_eligible
                )
            ),

        "stable_rows":
            int(
                len(
                    page_stable
                )
            ),

        "channel_count":
            int(
                len(
                    CACHED_CHANNEL_DTYPES
                )
            ),

        "patch_checks":
            int(
                len(
                    np.unique(
                        eligible_positions
                    )
                )
            ),

        "failures":
            int(
                page_failure_count
            ),

        "status":
            (
                "PASS"

                if page_failure_count
                == 0

                else "FAIL"
            ),
    })


    if (
        page_failure_count
        == 0
    ):

        print(
            f"[{page_number:03d}/"
            f"{len(page_list):03d}] "
            f"{page}: PASS | "
            f"shape={page_shape} | "
            f"eligible={len(page_eligible):,}"
        )


    else:

        print(
            f"[{page_number:03d}/"
            f"{len(page_list):03d}] "
            f"{page}: FAIL | "
            f"{page_failure_count} issue(s)"
        )


    del page_eligible
    del page_stable
    del eligible_y_page
    del eligible_x_page
    del in_bounds
    del eligible_positions

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# Durable preflight evidence
# ============================================================

channel_audit_df = pd.DataFrame(
    audit_rows
)


failure_df = pd.DataFrame(
    failure_rows,
    columns=[
        "page",
        "channel",
        "failure",
        "detail",
    ],
)


page_audit_df = pd.DataFrame(
    page_rows
)


channel_audit_path = (
    AUDIT
    / "observable_channel_preflight.csv"
)


failure_path = (
    AUDIT
    / "observable_channel_preflight_failures.csv"
)


page_audit_path = (
    AUDIT
    / "observable_channel_preflight_by_page.csv"
)


channel_audit_df.to_csv(
    channel_audit_path,
    index=False,
)


failure_df.to_csv(
    failure_path,
    index=False,
)


page_audit_df.to_csv(
    page_audit_path,
    index=False,
)


# ============================================================
# Aggregate diagnostics
# ============================================================

total_nonfinite = (
    int(
        channel_audit_df[
            "nonfinite"
        ].sum()
    )

    if len(
        channel_audit_df
    )

    else 0
)


total_probability_range_failures = (
    int(
        (
            channel_audit_df[
                [
                    "below_zero",
                    "above_one",
                ]
            ].sum()
        ).sum()
    )

    if len(
        channel_audit_df
    )

    else 0
)


total_nonbinary = (
    int(
        channel_audit_df[
            "nonbinary"
        ].sum()
    )

    if len(
        channel_audit_df
    )

    else 0
)


failed_pages = (
    page_audit_df.loc[
        page_audit_df[
            "status"
        ]
        == "FAIL",
        "page",
    ].tolist()
)


# ============================================================
# Explicit p_print / p_annotation report
# ============================================================

probability_audit = (
    channel_audit_df[
        channel_audit_df[
            "channel"
        ].isin(
            [
                "p_annotation",
                "p_print",
            ]
        )
    ]
    .copy()
)


probability_audit.to_csv(
    AUDIT
    / "probability_channel_preflight.csv",
    index=False,
)


if len(
    probability_audit
):

    print()
    print(
        "Probability-channel summary:"
    )

    print(
        probability_audit.groupby(
            "channel",
            as_index=False,
        ).agg(
            pages=(
                "page",
                "nunique",
            ),

            pixels=(
                "pixels",
                "sum",
            ),

            nonfinite=(
                "nonfinite",
                "sum",
            ),

            below_zero=(
                "below_zero",
                "sum",
            ),

            above_one=(
                "above_one",
                "sum",
            ),

            minimum=(
                "minimum",
                "min",
            ),

            maximum=(
                "maximum",
                "max",
            ),
        ).to_string(
            index=False
        )
    )


# ============================================================
# Final verdict
# ============================================================

preflight_summary = {
    "pages":
        int(
            len(
                page_audit_df
            )
        ),

    "channels_audited":
        int(
            len(
                channel_audit_df
            )
        ),

    "eligible_rows":
        int(
            len(
                eligible
            )
        ),

    "stable_rows":
        int(
            len(
                stable
            )
        ),

    "nonfinite_values":
        total_nonfinite,

    "probability_range_failures":
        total_probability_range_failures,

    "nonbinary_values":
        total_nonbinary,

    "failure_records":
        int(
            len(
                failure_df
            )
        ),

    "failed_pages":
        failed_pages,

    "row_block":
        int(
            PREFLIGHT_ROW_BLOCK
        ),

    "representative_patch_checks_per_page":
        int(
            PREFLIGHT_PATCHES_PER_PAGE
        ),

    "dataloader_workers":
        0,

    "verdict":
        (
            "PASS"
            if len(
                failure_df
            )
            == 0

            else "FAIL"
        ),
}


(
    AUDIT
    / "observable_channel_preflight_summary.json"
).write_text(
    json.dumps(
        preflight_summary,
        indent=2,
    )
)


print()
print("=" * 72)
print(
    "A5 OBSERVABLE-CHANNEL PREFLIGHT"
)
print("=" * 72)

print(
    json.dumps(
        preflight_summary,
        indent=2,
    )
)

print()
print(
    "Channel audit:",
    channel_audit_path,
)

print(
    "Page audit:",
    page_audit_path,
)

print(
    "Failure table:",
    failure_path,
)


# ============================================================
# FAIL CLOSED
# ============================================================

require(
    len(
        failure_df
    )
    == 0,
    (
        "A5 preflight FAILED. "
        f"{len(failure_df)} issue(s) were found across "
        f"{len(failed_pages)} page(s). "
        f"Do not start OOF training. See {failure_path}"
    ),
)


require(
    total_nonfinite
    == 0,
    (
        "A5 preflight found non-finite observable "
        f"values: {total_nonfinite}"
    ),
)


require(
    total_probability_range_failures
    == 0,
    (
        "A5 preflight found probability values "
        "outside [0,1]: "
        f"{total_probability_range_failures}"
    ),
)


require(
    total_nonbinary
    == 0,
    (
        "A5 preflight found non-binary values "
        "in binary channels: "
        f"{total_nonbinary}"
    ),
)


print()
print(
    "PASS: all A5 observable channels are finite, "
    "shape-compatible and schema-compatible."
)

print(
    "PASS: p_annotation and p_print are finite "
    "and bounded to [0,1]."
)

print(
    "PASS: all eligible coordinates remain inside "
    "the AMBIGUOUS population."
)

print(
    "PASS: representative 33x33 tensors were "
    "constructed successfully for every page."
)

print(
    "PASS: OOF training may proceed."
)

# 7. Source-group OOF model training and all-pixel scoring

In [ ]:
# @title
import gc
import time


# ------------------------------------------------------------
# Compact fold arrays once
# ------------------------------------------------------------

stable_fold_array = stable[
    "fold"
].to_numpy(
    dtype=np.int16,
    copy=True,
)

eligible_fold_array = eligible[
    "fold"
].to_numpy(
    dtype=np.int16,
    copy=True,
)


fold_ids = sorted(
    np.unique(
        stable_fold_array
    ).tolist()
)


# ------------------------------------------------------------
# OOF outputs
#
# float64 retained intentionally because isotonic calibration
# and durable OOF evidence use these values directly.
# ------------------------------------------------------------

raw_stable = np.full(
    len(stable),
    np.nan,
    dtype=np.float64,
)

raw_eligible = np.full(
    len(eligible),
    np.nan,
    dtype=np.float64,
)


model_rows = []

history_rows = []


# ============================================================
# Fold loop
# ============================================================

for fold_id in fold_ids:

    print()
    print("=" * 80)
    print("FOLD", fold_id)


    train_idx = np.flatnonzero(
        stable_fold_array
        != fold_id
    )

    test_idx = np.flatnonzero(
        stable_fold_array
        == fold_id
    )

    eligible_idx = np.flatnonzero(
        eligible_fold_array
        == fold_id
    )


    require(
        len(train_idx) > 0,
        f"Fold {fold_id}: empty training side",
    )

    require(
        len(test_idx) > 0,
        f"Fold {fold_id}: empty stable test side",
    )


    train_labels = stable_label[
        train_idx
    ]


    require(
        train_labels.min() == 0
        and train_labels.max() == 1,
        (
            f"Fold {fold_id}: "
            "training side lacks a class"
        ),
    )


    del train_labels

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    started = time.time()


    # --------------------------------------------------------
    # Train exactly one fold model
    # --------------------------------------------------------

    model, history = fit_model(
        train_idx
    )


    # --------------------------------------------------------
    # Score held-out stable rows
    # --------------------------------------------------------

    raw_stable[
        test_idx
    ] = predict_annotation(
        model,
        test_idx,
        stable_pid,
        stable_y,
        stable_x,
        stable_label,
    )


    # --------------------------------------------------------
    # Score COMPLETE eligible held-out population for this fold
    # --------------------------------------------------------

    if len(eligible_idx):

        raw_eligible[
            eligible_idx
        ] = predict_annotation(
            model,
            eligible_idx,
            eligible_pid,
            eligible_y,
            eligible_x,
            eligible_label,
        )


    # --------------------------------------------------------
    # Persist model immediately
    # --------------------------------------------------------

    checkpoint = (
        MODELS
        / (
            f"fold{int(fold_id)}_"
            f"seed{SEED}_"
            "a5_context_cnn.pt"
        )
    )


    # Save CPU state only.
    #
    # This avoids torch.save retaining references to active GPU
    # storage during serialisation.
    cpu_state = {
        key:
            value.detach().cpu()

        for key, value
        in model.state_dict().items()
    }


    torch.save(
        cpu_state,
        checkpoint,
    )


    checkpoint_sha = sha256_file(
        checkpoint
    )


    model_rows.append({
        "fold":
            int(
                fold_id
            ),

        "checkpoint":
            str(
                checkpoint
            ),

        "checkpoint_sha256":
            checkpoint_sha,

        "train_rows":
            int(
                len(
                    train_idx
                )
            ),

        "stable_test_rows":
            int(
                len(
                    test_idx
                )
            ),

        "eligible_test_rows":
            int(
                len(
                    eligible_idx
                )
            ),

        "seconds":
            float(
                time.time()
                - started
            ),
    })


    for epoch, loss in enumerate(
        history,
        start=1,
    ):

        history_rows.append({
            "fold":
                int(
                    fold_id
                ),

            "epoch":
                int(
                    epoch
                ),

            "mean_loss":
                float(
                    loss
                ),
        })


    # --------------------------------------------------------
    # Fold-local integrity BEFORE releasing model
    # --------------------------------------------------------

    require(
        np.isfinite(
            raw_stable[
                test_idx
            ]
        ).all(),
        (
            f"Fold {fold_id}: "
            "non-finite stable OOF predictions"
        ),
    )


    if len(eligible_idx):

        require(
            np.isfinite(
                raw_eligible[
                    eligible_idx
                ]
            ).all(),
            (
                f"Fold {fold_id}: "
                "non-finite eligible OOF predictions"
            ),
        )


    print(
        "  stable scored:",
        f"{len(test_idx):,}",
    )

    print(
        "  eligible scored:",
        f"{len(eligible_idx):,}",
    )

    print(
        "  checkpoint:",
        checkpoint.name,
    )


    # --------------------------------------------------------
    # Aggressive fold cleanup
    # --------------------------------------------------------

    model.to(
        "cpu"
    )


    del model
    del cpu_state
    del history

    del train_idx
    del test_idx
    del eligible_idx


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


# ============================================================
# Complete OOF integrity
# ============================================================

require(
    np.isfinite(
        raw_stable
    ).all(),
    "Incomplete stable OOF predictions.",
)


require(
    np.isfinite(
        raw_eligible
    ).all(),
    "Incomplete all-eligible OOF predictions.",
)


# ============================================================
# Persist complete OOF score vectors
# ============================================================

np.save(
    TABLES
    / "stable_raw_oof.npy",
    raw_stable,
)


np.save(
    TABLES
    / "eligible_raw_oof.npy",
    raw_eligible,
)


pd.DataFrame(
    model_rows
).to_csv(
    AUDIT
    / "model_inventory.csv",
    index=False,
)


pd.DataFrame(
    history_rows
).to_csv(
    TABLES
    / "training_history.csv",
    index=False,
)


print()
print("=" * 80)
print("OOF TRAINING COMPLETE")
print("=" * 80)

print(
    "Models:",
    len(
        model_rows
    ),
)

print(
    "stable OOF:",
    len(
        raw_stable
    ),
)

print(
    "eligible OOF:",
    len(
        raw_eligible
    ),
)

# 8. Fold-specific cross-fitted isotonic calibration

In [ ]:
# @title
import gc


p_stable = np.full(
    len(stable),
    np.nan,
    dtype=np.float64,
)

p_eligible = np.full(
    len(eligible),
    np.nan,
    dtype=np.float64,
)


calibrator_rows = []


for fold_id in fold_ids:

    train_idx = np.flatnonzero(
        stable_fold_array
        != fold_id
    )

    test_idx = np.flatnonzero(
        stable_fold_array
        == fold_id
    )

    eligible_idx = np.flatnonzero(
        eligible_fold_array
        == fold_id
    )


    # --------------------------------------------------------
    # Fit only on stable rows outside held-out source groups
    # --------------------------------------------------------

    calibrator = IsotonicRegression(
        out_of_bounds="clip"
    )


    calibrator.fit(
        raw_stable[
            train_idx
        ],
        stable_label[
            train_idx
        ],
    )


    # --------------------------------------------------------
    # Held-out stable calibration
    # --------------------------------------------------------

    p_stable[
        test_idx
    ] = calibrator.predict(
        raw_stable[
            test_idx
        ]
    )


    # --------------------------------------------------------
    # Complete held-out eligible population
    # --------------------------------------------------------

    if len(
        eligible_idx
    ):

        p_eligible[
            eligible_idx
        ] = calibrator.predict(
            raw_eligible[
                eligible_idx
            ]
        )


    # --------------------------------------------------------
    # Persist calibrator
    # --------------------------------------------------------

    path = (
        CALIBRATORS
        / (
            f"fold{int(fold_id)}_"
            f"seed{SEED}_"
            "annotation_isotonic.joblib"
        )
    )


    joblib.dump(
        calibrator,
        path,
    )


    payload = {
        "x":
            calibrator
            .X_thresholds_
            .tolist(),

        "y":
            calibrator
            .y_thresholds_
            .tolist(),
    }


    sidecar = {
        "fold":
            int(
                fold_id
            ),

        "target":
            "ANNOTATION_SAFE_TO_CLEAN",

        "method":
            "IsotonicRegression",

        "training_rule":
            (
                "stable OOF rows outside "
                "held-out fold"
            ),

        "semantic_sha256":
            object_hash(
                payload
            ),

        "joblib_sha256":
            sha256_file(
                path
            ),

        **payload,
    }


    sidecar_path = (
        CALIBRATORS
        / (
            f"fold{int(fold_id)}_"
            f"seed{SEED}_"
            "annotation_isotonic.json"
        )
    )


    sidecar_path.write_text(
        json.dumps(
            sidecar,
            indent=2,
        )
    )


    # --------------------------------------------------------
    # Exact durable reload replay
    # --------------------------------------------------------

    replay_calibrator = joblib.load(
        path
    )


    if len(
        eligible_idx
    ):

        replay = (
            replay_calibrator
            .predict(
                raw_eligible[
                    eligible_idx
                ]
            )
            .astype(
                np.float64,
                copy=False,
            )
        )


        require(
            np.array_equal(
                replay,
                p_eligible[
                    eligible_idx
                ],
            ),
            (
                f"Fold {fold_id}: "
                "calibrator reload mismatch"
            ),
        )


        del replay


    calibrator_rows.append({
        "fold":
            int(
                fold_id
            ),

        "joblib":
            str(
                path
            ),

        "joblib_sha256":
            sha256_file(
                path
            ),

        "semantic_sha256":
            object_hash(
                payload
            ),

        "exact_reload_replay":
            True,
    })


    # --------------------------------------------------------
    # Fold cleanup
    # --------------------------------------------------------

    del calibrator
    del replay_calibrator
    del payload
    del sidecar

    del train_idx
    del test_idx
    del eligible_idx


    gc.collect()


    print(
        f"Fold {fold_id}: "
        "calibrator persisted and replayed"
    )


# ============================================================
# Complete calibration integrity
# ============================================================

require(
    np.isfinite(
        p_stable
    ).all(),
    (
        "Incomplete stable calibrated "
        "probabilities."
    ),
)


require(
    np.isfinite(
        p_eligible
    ).all(),
    (
        "Incomplete eligible calibrated "
        "probabilities."
    ),
)


# ============================================================
# Persist calibrated evidence
# ============================================================

np.save(
    TABLES
    / "stable_calibrated_oof.npy",
    p_stable,
)


np.save(
    TABLES
    / "eligible_calibrated_oof.npy",
    p_eligible,
)


pd.DataFrame(
    calibrator_rows
).to_csv(
    AUDIT
    / "calibrator_inventory.csv",
    index=False,
)


print(
    "Calibrators persisted:",
    len(
        calibrator_rows
    ),
)

# 9. Conservative CLEAN/KEEP/UNCERTAIN policy and threshold sweep

MEMORY-SAFE VERSION

Key changes:
  - Do not retain one complete policy DataFrame per threshold.
  - Retain detailed predictions only for the primary policy.
  - Write page/component results immediately by threshold.
  - Avoid Python sets of full-page coordinates.


In [ ]:
# @title
import gc
import shutil


# ============================================================
# Prediction table
# ============================================================

predictions = eligible.copy()

predictions[
    "raw_p_annotation"
] = raw_eligible

predictions[
    "p_annotation_a5"
] = p_eligible


require(
    np.isfinite(
        predictions[
            "raw_p_annotation"
        ]
    ).all(),
    "Raw A5 predictions contain non-finite values.",
)

require(
    np.isfinite(
        predictions[
            "p_annotation_a5"
        ]
    ).all(),
    "Calibrated A5 probabilities contain non-finite values.",
)


# Compact frequently used arrays once.
prediction_probability = predictions[
    "p_annotation_a5"
].to_numpy(
    dtype=np.float64,
    copy=False,
)

prediction_page = predictions[
    "page"
].astype(str)


# ============================================================
# Threshold-sweep shard directories
# ============================================================

SWEEP_SHARDS = (
    TABLES
    / "a5_threshold_sweep_shards"
)

SWEEP_PAGE_SHARDS = (
    SWEEP_SHARDS
    / "pages"
)

SWEEP_COMPONENT_SHARDS = (
    SWEEP_SHARDS
    / "components"
)


if SWEEP_SHARDS.exists():

    shutil.rmtree(
        SWEEP_SHARDS
    )


SWEEP_PAGE_SHARDS.mkdir(
    parents=True,
    exist_ok=False,
)

SWEEP_COMPONENT_SHARDS.mkdir(
    parents=True,
    exist_ok=False,
)


# ============================================================
# Decisions
# ============================================================

def decisions_at_threshold(
    probability,
    threshold,
):

    """
    Only extreme annotation probability receives CLEAN.

    Extreme PRINT/KEEP probability receives explicit KEEP.

    Everything else remains UNCERTAIN and therefore falls back
    to exact A1.
    """

    probability = np.asarray(
        probability,
        dtype=np.float64,
    )


    final = np.full(
        len(
            probability
        ),
        "UNCERTAIN",
        dtype=object,
    )


    final[
        probability
        <= (
            1.0
            - threshold
        )
    ] = "KEEP"


    final[
        probability
        >= threshold
    ] = "CLEAN"


    return final


# ============================================================
# Page-global minimum-run filter
#
# This uses component IDs at selected coordinates rather than
# constructing Python sets for every singleton component.
# ============================================================

def apply_minimum_run(
    frame,
    provisional,
    minimum_run=2,
):

    final = np.asarray(
        provisional,
        dtype=object,
    ).copy()


    demoted = 0


    for page, group in frame.groupby(
        "page",
        sort=False,
        observed=True,
    ):

        group_indices = group.index.to_numpy(
            dtype=np.int64,
            copy=False,
        )


        selected_local = (
            provisional[
                group_indices
            ]
            == "CLEAN"
        )


        if not selected_local.any():
            continue


        selected_indices = group_indices[
            selected_local
        ]


        shape = runtime[
            str(page)
        ][
            "observed"
        ].shape


        selection_mask = np.zeros(
            shape,
            dtype=np.uint8,
        )


        selected_y = frame.loc[
            selected_indices,
            "y",
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )

        selected_x = frame.loc[
            selected_indices,
            "x",
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )


        selection_mask[
            selected_y,
            selected_x,
        ] = 1


        (
            component_count,
            component_labels,
            component_stats,
            _,
        ) = cv2.connectedComponentsWithStats(
            selection_mask,
            8,
        )


        selected_component_ids = component_labels[
            selected_y,
            selected_x,
        ]


        selected_component_sizes = component_stats[
            selected_component_ids,
            cv2.CC_STAT_AREA,
        ]


        demote_local = (
            selected_component_sizes
            < minimum_run
        )


        if demote_local.any():

            demoted_indices = selected_indices[
                demote_local
            ]

            final[
                demoted_indices
            ] = "UNCERTAIN"

            demoted += int(
                demote_local.sum()
            )


        del selection_mask
        del component_labels
        del component_stats
        del selected_y
        del selected_x
        del selected_component_ids
        del selected_component_sizes
        del selected_indices
        del group_indices

        gc.collect()


    return (
        final,
        demoted,
    )


# ============================================================
# Policy metrics
# ============================================================

def policy_metrics(
    frame,
    decisions,
    threshold,
    demotions,
):

    # Copy only columns required for policy analysis.
    work = frame[
        [
            "page",
            "source_group",
            "y",
            "x",
            "label",
            "candidate_id",
            "fold",
            "raw_p_annotation",
            "p_annotation_a5",
        ]
    ].copy()


    work[
        "decision"
    ] = decisions


    clean = (
        work[
            "decision"
        ]
        == "CLEAN"
    )


    tp_cleanup = int(
        (
            clean
            & (
                work[
                    "label"
                ]
                == LABEL_ANNOTATION
            )
        ).sum()
    )


    print_loss = int(
        (
            clean
            & (
                work[
                    "label"
                ]
                == LABEL_PRINT
            )
        ).sum()
    )


    uncertain_changed = int(
        (
            clean
            & (
                work[
                    "label"
                ]
                == LABEL_UNCERTAIN
            )
        ).sum()
    )


    # --------------------------------------------------------
    # Connected components of selected cleanup, by reviewed
    # authority component
    # --------------------------------------------------------

    component_rows = []


    clean_rows = work.loc[
        clean
    ]


    for page, group in clean_rows.groupby(
        "page",
        sort=False,
        observed=True,
    ):

        page = str(
            page
        )


        authority = (
            runtime[
                page
            ][
                "reviewed_mask"
            ]
            > 0
        )


        (
            component_count,
            component_labels,
            component_stats,
            _,
        ) = cv2.connectedComponentsWithStats(
            authority.astype(
                np.uint8
            ),
            8,
        )


        ys = group[
            "y"
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )

        xs = group[
            "x"
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )


        component_ids = component_labels[
            ys,
            xs,
        ]


        component_group = group.copy()

        component_group[
            "component_id"
        ] = component_ids


        for component_id, selected_component in component_group[
            component_group[
                "component_id"
            ]
            > 0
        ].groupby(
            "component_id",
            sort=False,
        ):

            annotation_count = int(
                (
                    selected_component[
                        "label"
                    ]
                    == LABEL_ANNOTATION
                ).sum()
            )

            print_count = int(
                (
                    selected_component[
                        "label"
                    ]
                    == LABEL_PRINT
                ).sum()
            )

            uncertain_count = int(
                (
                    selected_component[
                        "label"
                    ]
                    == LABEL_UNCERTAIN
                ).sum()
            )


            catastrophic = bool(
                print_count >= 50
                or (
                    print_count
                    / max(
                        annotation_count,
                        1,
                    )
                    > 1.0
                    and print_count >= 20
                )
            )


            component_rows.append({
                "threshold":
                    float(
                        threshold
                    ),

                "page":
                    page,

                "source_group":
                    str(
                        selected_component[
                            "source_group"
                        ].iloc[
                            0
                        ]
                    ),

                "component_id":
                    int(
                        component_id
                    ),

                "authority_component_area":
                    int(
                        component_stats[
                            int(
                                component_id
                            ),
                            cv2.CC_STAT_AREA,
                        ]
                    ),

                "selected":
                    int(
                        len(
                            selected_component
                        )
                    ),

                "annotation_cleaned":
                    annotation_count,

                "print_lost":
                    print_count,

                "truth_uncertain_changed":
                    uncertain_count,

                "catastrophic":
                    catastrophic,
            })


        del authority
        del component_labels
        del component_stats
        del component_ids
        del component_group
        del ys
        del xs

        gc.collect()


    component_columns = [
        "threshold",
        "page",
        "source_group",
        "component_id",
        "authority_component_area",
        "selected",
        "annotation_cleaned",
        "print_lost",
        "truth_uncertain_changed",
        "catastrophic",
    ]


    components = pd.DataFrame(
        component_rows,
        columns=component_columns,
    )


    catastrophes = (
        int(
            components[
                "catastrophic"
            ].sum()
        )
        if len(
            components
        )
        else 0
    )


    # --------------------------------------------------------
    # Page results
    # --------------------------------------------------------

    page_rows = []


    for page, group in work.groupby(
        "page",
        sort=False,
        observed=True,
    ):

        selected = (
            group[
                "decision"
            ]
            == "CLEAN"
        )


        annotation_count = int(
            (
                selected
                & (
                    group[
                        "label"
                    ]
                    == LABEL_ANNOTATION
                )
            ).sum()
        )


        print_count = int(
            (
                selected
                & (
                    group[
                        "label"
                    ]
                    == LABEL_PRINT
                )
            ).sum()
        )


        uncertain_count = int(
            (
                selected
                & (
                    group[
                        "label"
                    ]
                    == LABEL_UNCERTAIN
                )
            ).sum()
        )


        if len(
            components
        ):

            page_catastrophes = int(
                (
                    (
                        components[
                            "page"
                        ]
                        == str(
                            page
                        )
                    )
                    & components[
                        "catastrophic"
                    ]
                ).sum()
            )

        else:

            page_catastrophes = 0


        page_rows.append({
            "threshold":
                float(
                    threshold
                ),

            "page":
                str(
                    page
                ),

            "source_group":
                str(
                    group[
                        "source_group"
                    ].iloc[
                        0
                    ]
                ),

            "eligible":
                int(
                    len(
                        group
                    )
                ),

            "selected_clean":
                int(
                    selected.sum()
                ),

            "annotation_cleaned":
                annotation_count,

            "print_lost":
                print_count,

            "truth_uncertain_changed":
                uncertain_count,

            "utility":
                (
                    annotation_count
                    - 5
                    * print_count
                    - 500
                    * page_catastrophes
                ),

            "catastrophic_components":
                page_catastrophes,

            "clean_coverage":
                float(
                    selected.mean()
                ),

            "uncertain_rate":
                float(
                    (
                        group[
                            "decision"
                        ]
                        == "UNCERTAIN"
                    ).mean()
                ),
        })


    pages = pd.DataFrame(
        page_rows
    )


    # --------------------------------------------------------
    # Page-clustered bootstrap
    # --------------------------------------------------------

    rng = np.random.default_rng(
        SEED
    )


    page_ids = pages[
        "page"
    ].to_numpy()


    indexed = pages.set_index(
        "page"
    )


    bootstrap = np.empty(
        (
            BOOTSTRAP_RESAMPLES,
            3,
        ),
        dtype=np.float64,
    )


    for bootstrap_index in range(
        BOOTSTRAP_RESAMPLES
    ):

        sample = indexed.loc[
            rng.choice(
                page_ids,
                size=len(
                    page_ids
                ),
                replace=True,
            )
        ]


        annotation_total = int(
            sample[
                "annotation_cleaned"
            ].sum()
        )

        print_total = int(
            sample[
                "print_lost"
            ].sum()
        )

        catastrophe_total = int(
            sample[
                "catastrophic_components"
            ].sum()
        )


        bootstrap[
            bootstrap_index,
            0,
        ] = (
            annotation_total
            - 5
            * print_total
            - 500
            * catastrophe_total
        )


        bootstrap[
            bootstrap_index,
            1,
        ] = (
            print_total
            / max(
                annotation_total,
                1,
            )
        )


        bootstrap[
            bootstrap_index,
            2,
        ] = (
            print_total
            / max(
                annotation_total
                + print_total,
                1,
            )
        )


    summary = {
        "threshold":
            float(
                threshold
            ),

        "eligible_pixels":
            int(
                len(
                    work
                )
            ),

        "selected_clean_pixels":
            int(
                clean.sum()
            ),

        "annotation_cleaned":
            tp_cleanup,

        "print_lost":
            print_loss,

        "truth_uncertain_changed":
            uncertain_changed,

        "print_loss_per_annotation_cleaned":
            print_loss
            / max(
                tp_cleanup,
                1,
            ),

        "false_cleanup_rate":
            print_loss
            / max(
                tp_cleanup
                + print_loss,
                1,
            ),

        "utility":
            (
                tp_cleanup
                - 5
                * print_loss
                - 500
                * catastrophes
            ),

        "catastrophic_components":
            catastrophes,

        "benefiting_pages":
            int(
                (
                    pages[
                        "annotation_cleaned"
                    ]
                    >= 10
                ).sum()
            ),

        "benefiting_source_groups":
            int(
                pages.loc[
                    pages[
                        "annotation_cleaned"
                    ]
                    >= 10,
                    "source_group",
                ].nunique()
            ),

        "clean_coverage":
            float(
                clean.mean()
            ),

        "uncertain_rate":
            float(
                (
                    work[
                        "decision"
                    ]
                    == "UNCERTAIN"
                ).mean()
            ),

        "minimum_run_demotions":
            int(
                demotions
            ),

        "bootstrap_utility_95_low":
            float(
                np.quantile(
                    bootstrap[
                        :,
                        0,
                    ],
                    0.025,
                )
            ),

        "bootstrap_utility_95_high":
            float(
                np.quantile(
                    bootstrap[
                        :,
                        0,
                    ],
                    0.975,
                )
            ),

        "bootstrap_print_loss_per_annotation_95_high":
            float(
                np.quantile(
                    bootstrap[
                        :,
                        1,
                    ],
                    0.975,
                )
            ),

        "bootstrap_false_cleanup_95_high":
            float(
                np.quantile(
                    bootstrap[
                        :,
                        2,
                    ],
                    0.975,
                )
            ),
    }


    del clean_rows
    del bootstrap
    del indexed

    gc.collect()


    return (
        summary,
        pages,
        components,
        work,
    )


# ============================================================
# Threshold sweep
# ============================================================

sweep_summaries = []

page_shard_paths = []

component_shard_paths = []

primary = None


for threshold_index, threshold in enumerate(
    CLEAN_THRESHOLDS,
    start=1,
):

    threshold = float(
        threshold
    )


    print(
        f"[{threshold_index}/{len(CLEAN_THRESHOLDS)}] "
        f"threshold={threshold:.6f}"
    )


    provisional = decisions_at_threshold(
        prediction_probability,
        threshold,
    )


    final, demotions = apply_minimum_run(
        predictions,
        provisional,
        MIN_CLEAN_RUN,
    )


    (
        summary,
        pages_df,
        components_df,
        work,
    ) = policy_metrics(
        predictions,
        final,
        threshold,
        demotions,
    )


    sweep_summaries.append(
        summary
    )


    threshold_token = (
        f"{threshold:.6f}"
        .replace(
            ".",
            "_",
        )
    )


    page_shard = (
        SWEEP_PAGE_SHARDS
        / (
            f"threshold_"
            f"{threshold_token}.parquet"
        )
    )


    pages_df.to_parquet(
        page_shard,
        index=False,
    )


    page_shard_paths.append(
        page_shard
    )


    if len(
        components_df
    ):

        component_shard = (
            SWEEP_COMPONENT_SHARDS
            / (
                f"threshold_"
                f"{threshold_token}.parquet"
            )
        )


        components_df.to_parquet(
            component_shard,
            index=False,
        )


        component_shard_paths.append(
            component_shard
        )


    # Retain only the primary complete pixel table.
    if np.isclose(
        threshold,
        float(
            PRIMARY_CLEAN_THRESHOLD
        ),
        rtol=0.0,
        atol=1e-12,
    ):

        require(
            primary is None,
            (
                "PRIMARY_CLEAN_THRESHOLD matched "
                "more than one threshold."
            ),
        )

        primary = work


    else:

        del work


    del provisional
    del final
    del pages_df
    del components_df

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Complete sweep outputs
# ============================================================

require(
    primary is not None,
    (
        "PRIMARY_CLEAN_THRESHOLD was not found in "
        "CLEAN_THRESHOLDS."
    ),
)


sweep = (
    pd.DataFrame(
        sweep_summaries
    )
    .sort_values(
        "threshold",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


sweep.to_csv(
    TABLES
    / "a5_threshold_sweep.csv",
    index=False,
)


# These are small page/component summaries, not pixel tables.
sweep_pages = pd.concat(
    [
        pd.read_parquet(
            path
        )

        for path
        in page_shard_paths
    ],
    ignore_index=True,
)


sweep_pages.to_csv(
    TABLES
    / "a5_threshold_sweep_by_page.csv",
    index=False,
)


del sweep_pages

gc.collect()


if component_shard_paths:

    sweep_components = pd.concat(
        [
            pd.read_parquet(
                path
            )

            for path
            in component_shard_paths
        ],
        ignore_index=True,
    )

else:

    sweep_components = pd.DataFrame()


sweep_components.to_csv(
    TABLES
    / "a5_threshold_sweep_components.csv",
    index=False,
)


del sweep_components

gc.collect()


# ============================================================
# Primary prediction record
# ============================================================

primary.to_parquet(
    TABLES
    / "a5_primary_oof_predictions.parquet",
    index=False,
)


# CSV can be large. It is retained because the original
# experiment requested both durable formats.
primary.to_csv(
    TABLES
    / "a5_primary_oof_predictions.csv",
    index=False,
)


print(
    sweep.to_string(
        index=False
    )
)

# 10. Assemble complete A5 OOF pages from exact A1 fallback

MEMORY-SAFE VERSION

One page is copied, assembled, verified, written and released
before the next page is loaded.


In [ ]:
# @title
import gc


primary_page_metrics = []


for page_number, (
    page,
    group,
) in enumerate(
    primary.groupby(
        "page",
        sort=False,
        observed=True,
    ),
    start=1,
):

    page = str(
        page
    )


    page_runtime = runtime[
        page
    ]


    # These remain read-only disk-backed arrays.
    observed = page_runtime[
        "observed"
    ]

    a1_source = page_runtime[
        "a1"
    ]

    paper = page_runtime[
        "paper"
    ]

    authority = (
        page_runtime[
            "reviewed_mask"
        ]
        > 0
    )


    expected_shape = observed.shape


    for name, array in {
        "A1":
            a1_source,

        "paper":
            paper,

        "authority":
            authority,
    }.items():

        require(
            array.shape
            == expected_shape,
            (
                f"{page}: {name} shape "
                f"{array.shape} != observed "
                f"{expected_shape}"
            ),
        )


    # --------------------------------------------------------
    # Build only this page's CLEAN mask
    # --------------------------------------------------------

    clean_rows = group[
        group[
            "decision"
        ]
        == "CLEAN"
    ]


    clean_mask = np.zeros(
        expected_shape,
        dtype=bool,
    )


    if len(
        clean_rows
    ):

        clean_y = clean_rows[
            "y"
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )

        clean_x = clean_rows[
            "x"
        ].to_numpy(
            dtype=np.int32,
            copy=False,
        )


        clean_mask[
            clean_y,
            clean_x,
        ] = True


        del clean_y
        del clean_x


    require(
        not np.any(
            clean_mask
            & ~authority
        ),
        (
            f"{page}: A5 CLEAN decision escapes "
            "reviewed authority"
        ),
    )


    # --------------------------------------------------------
    # A5 is the only mutable full-page array.
    # --------------------------------------------------------

    a5 = np.array(
        a1_source,
        dtype=np.uint8,
        copy=True,
    )


    a5[
        clean_mask
    ] = paper[
        clean_mask
    ]


    # --------------------------------------------------------
    # Exact A1 fallback/integrity
    # --------------------------------------------------------

    outside = int(
        np.sum(
            a5[
                ~authority
            ]
            != observed[
                ~authority
            ]
        )
    )


    nonclean = int(
        np.sum(
            a5[
                ~clean_mask
            ]
            != a1_source[
                ~clean_mask
            ]
        )
    )


    selected_paper_mismatch = int(
        np.sum(
            a5[
                clean_mask
            ]
            != paper[
                clean_mask
            ]
        )
    )


    require(
        outside == 0,
        (
            f"{page}: outside-mask "
            f"drift={outside}"
        ),
    )


    require(
        nonclean == 0,
        (
            f"{page}: non-CLEAN regression "
            f"from A1={nonclean}"
        ),
    )


    require(
        selected_paper_mismatch
        == 0,
        (
            f"{page}: selected CLEAN pixels "
            "do not equal the frozen paper estimate: "
            f"{selected_paper_mismatch}"
        ),
    )


    # --------------------------------------------------------
    # Persist page
    # --------------------------------------------------------

    page_output = (
        PAGES
        / page
    )


    page_output.mkdir(
        parents=True,
        exist_ok=False,
    )


    require(
        cv2.imwrite(
            str(
                page_output
                / "observed.png"
            ),
            np.asarray(
                observed
            ),
        ),
        f"{page}: failed to write observed.png",
    )


    require(
        cv2.imwrite(
            str(
                page_output
                / "A1.png"
            ),
            np.asarray(
                a1_source
            ),
        ),
        f"{page}: failed to write A1.png",
    )


    require(
        cv2.imwrite(
            str(
                page_output
                / "A5.png"
            ),
            a5,
        ),
        f"{page}: failed to write A5.png",
    )


    require(
        cv2.imwrite(
            str(
                page_output
                / "clean_decision.png"
            ),
            clean_mask.astype(
                np.uint8
            )
            * 255,
        ),
        (
            f"{page}: failed to write "
            "clean_decision.png"
        ),
    )


    difference = np.zeros(
        expected_shape,
        dtype=np.uint8,
    )


    difference[
        a5
        != a1_source
    ] = 255


    require(
        cv2.imwrite(
            str(
                page_output
                / "A5_minus_A1.png"
            ),
            difference,
        ),
        (
            f"{page}: failed to write "
            "A5_minus_A1.png"
        ),
    )


    # --------------------------------------------------------
    # Page metric
    # --------------------------------------------------------

    metric = {
        "page":
            page,

        "source_group":
            str(
                group[
                    "source_group"
                ].iloc[
                    0
                ]
            ),

        "eligible":
            int(
                len(
                    group
                )
            ),

        "selected_clean":
            int(
                clean_mask.sum()
            ),

        "annotation_cleaned":
            int(
                (
                    (
                        group[
                            "decision"
                        ]
                        == "CLEAN"
                    )
                    & (
                        group[
                            "label"
                        ]
                        == LABEL_ANNOTATION
                    )
                ).sum()
            ),

        "print_lost":
            int(
                (
                    (
                        group[
                            "decision"
                        ]
                        == "CLEAN"
                    )
                    & (
                        group[
                            "label"
                        ]
                        == LABEL_PRINT
                    )
                ).sum()
            ),

        "truth_uncertain_changed":
            int(
                (
                    (
                        group[
                            "decision"
                        ]
                        == "CLEAN"
                    )
                    & (
                        group[
                            "label"
                        ]
                        == LABEL_UNCERTAIN
                    )
                ).sum()
            ),

        "outside_mask_drift":
            outside,

        "nonclean_a1_regression":
            nonclean,

        "selected_paper_mismatch":
            selected_paper_mismatch,

        "observed_sha256":
            sha256_file(
                page_output
                / "observed.png"
            ),

        "a1_sha256":
            sha256_file(
                page_output
                / "A1.png"
            ),

        "a5_sha256":
            sha256_file(
                page_output
                / "A5.png"
            ),

        "clean_decision_sha256":
            sha256_file(
                page_output
                / "clean_decision.png"
            ),
    }


    primary_page_metrics.append(
        metric
    )


    print(
        f"[{page_number:03d}/"
        f"{primary.page.nunique():03d}] "
        f"{page}: CLEAN="
        f"{int(clean_mask.sum()):,}"
    )


    # --------------------------------------------------------
    # Release the only mutable full-page arrays
    # --------------------------------------------------------

    del clean_rows
    del clean_mask
    del a5
    del difference
    del authority

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Page/source summaries
# ============================================================

primary_page_df = pd.DataFrame(
    primary_page_metrics
)


primary_page_df.to_csv(
    TABLES
    / "a5_primary_results_by_page.csv",
    index=False,
)


primary_group_df = (
    primary_page_df
    .groupby(
        "source_group",
        as_index=False,
    )
    .agg(
        pages=(
            "page",
            "nunique",
        ),

        eligible=(
            "eligible",
            "sum",
        ),

        selected_clean=(
            "selected_clean",
            "sum",
        ),

        annotation_cleaned=(
            "annotation_cleaned",
            "sum",
        ),

        print_lost=(
            "print_lost",
            "sum",
        ),

        truth_uncertain_changed=(
            "truth_uncertain_changed",
            "sum",
        ),
    )
)


primary_group_df[
    "print_loss_per_annotation_cleaned"
] = (
    primary_group_df[
        "print_lost"
    ]
    /
    primary_group_df[
        "annotation_cleaned"
    ].clip(
        lower=1
    )
)


primary_group_df.to_csv(
    TABLES
    / "a5_primary_results_by_source_group.csv",
    index=False,
)


print(
    "A5 whole pages:",
    len(
        primary_page_df
    ),
)

# 11. Visual audit queues and contact sheets

In [ ]:
# @title
rank = primary_page_df.copy()
rank['risk_score'] = rank.print_lost*1000 + rank.truth_uncertain_changed*100 + rank.annotation_cleaned
rank.sort_values('risk_score',ascending=False).to_csv(TABLES/'a5_visual_review_queue.csv',index=False)


def rgb(gray):
    return cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)


def overlay(gray, mask, colour, alpha=.55):
    base=rgb(gray); out=base.copy(); m=np.asarray(mask,bool); c=np.asarray(colour,np.float32)
    out[m]=np.clip((1-alpha)*base[m]+alpha*c,0,255).astype(np.uint8)
    return out


def contact_sheet(rows,name,max_pages=16):
    rows=rows.head(max_pages)
    if not len(rows):
        return
    fig,axes=plt.subplots(len(rows),4,figsize=(20,max(5,5*len(rows))),squeeze=False)
    for i,(_,row) in enumerate(rows.iterrows()):
        page=str(row.page); d=PAGES/page
        obs=read_gray(d/'observed.png'); a1=read_gray(d/'A1.png'); a5=read_gray(d/'A5.png')
        clean_mask=read_gray(d/'clean_decision.png')>0
        panels=[
            (rgb(obs),'Observed'),
            (rgb(a1),'A1 fallback'),
            (rgb(a5),'A5 candidate'),
            (overlay(obs,clean_mask,(255,215,0)),f'CLEAN selected: {int(clean_mask.sum()):,}'),
        ]
        for ax,(image,title) in zip(axes[i],panels):
            ax.imshow(image); ax.set_title(f'{page} — {title}'); ax.axis('off')
    plt.tight_layout(); plt.savefig(VISUALS/f'{name}.png',dpi=130,bbox_inches='tight'); plt.close(fig)

contact_sheet(rank.sort_values(['print_lost','selected_clean'],ascending=False),'most_print_loss')
contact_sheet(rank.sort_values(['annotation_cleaned','print_lost'],ascending=[False,True]),'most_annotation_cleanup')
contact_sheet(rank.sort_values('selected_clean',ascending=False),'most_changed')

plt.figure(figsize=(9,7))
plt.plot(sweep.print_loss_per_annotation_cleaned,sweep.clean_coverage,marker='o')
for row in sweep.itertuples():
    plt.text(row.print_loss_per_annotation_cleaned,row.clean_coverage,' '+str(row.threshold))
plt.xlabel('Print pixels lost per annotation pixel cleaned')
plt.ylabel('Eligible CLEAN coverage')
plt.title('A5 development safety/coverage frontier')
plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(VISUALS/'a5_safety_coverage_frontier.png',dpi=180); plt.close()

print('Visuals:',VISUALS)

# 12. Development verdict, identity and durable training bundle

In [ ]:
# @title
primary_summary = sweep.loc[sweep.threshold==PRIMARY_CLEAN_THRESHOLD].iloc[0].to_dict()

# Development evidence flags. These do not authorise promotion.
gates = {
    'integrity_finite_predictions': bool(np.isfinite(raw_eligible).all() and np.isfinite(p_eligible).all()),
    'source_group_oof_no_overlap': True,
    'outside_mask_immutability': bool((primary_page_df.outside_mask_drift==0).all()),
    'nonclean_a1_equivalence': bool((primary_page_df.nonclean_a1_regression==0).all()),
    'zero_catastrophic_components': bool(primary_summary['catastrophic_components']==0),
    'positive_utility': bool(primary_summary['utility']>0),
    'positive_clustered_utility': bool(primary_summary['bootstrap_utility_95_low']>0),
    'nontrivial_cleanup': bool(primary_summary['annotation_cleaned']>=500 and primary_summary['benefiting_pages']>=10),
    'visual_audit_complete': False,
}

first_failed = next((name for name,value in gates.items() if value is False),None)
verdict = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'88_A5_Contextual_Safe_Cleanup_Model_Training',
    'development_only':True,
    'independent_validation':False,
    'production_authorisation':False,
    'source_notebook87_run':str(SOURCE_N87),
    'primary_threshold':PRIMARY_CLEAN_THRESHOLD,
    'primary_summary':primary_summary,
    'gates':gates,
    'first_failed_or_incomplete_gate':first_failed,
    'decision':(
        'DEVELOPMENT_EVIDENCE_READY_FOR_VISUAL_REVIEW'
        if first_failed=='visual_audit_complete'
        else 'RETAIN_AS_DEVELOPMENT_RESEARCH'
    ),
}
(RUN/'notebook88_verdict.json').write_text(json.dumps(verdict,indent=2,default=str))

software={
    'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,
    'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,
    'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform(),
}
(RUN/'software_environment.json').write_text(json.dumps(software,indent=2))

identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'notebook88-a5-contextual-safe-cleanup-development',
    'source_notebook87_run':str(SOURCE_N87),
    'source_prediction_lock_sha256':sha256_file(SOURCE_N87/'prediction_lock.json'),
    'source_truth_table_sha256':sha256_file(SOURCE_N87/'tables'/'paired_hitl_ambiguous_truth.parquet'),
    'review_manifest_sha256':sha256_file(review_manifest_path),
    'channel_schema_sha256':sha256_file(INPUTS/'channel_schema.json'),
    'fold_map_sha256':sha256_file(INPUTS/'fold_map.csv'),
    'seed':SEED,'patch':PATCH,'channels':CHANNELS,'batch':BATCH,'epochs':EPOCHS,
    'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,
    'clean_thresholds':CLEAN_THRESHOLDS,'primary_clean_threshold':PRIMARY_CLEAN_THRESHOLD,
    'minimum_clean_run':MIN_CLEAN_RUN,'models':model_rows,'calibrators':calibrator_rows,
    'truth_in_inference_channels':False,'independent_data_used':False,
    'software':software,
}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

lines=[
    'NOTEBOOK 88 — A5 CONTEXTUAL SAFE-CLEANUP MODEL TRAINING','',
    f'Source Notebook 87 run: {SOURCE_N87}',
    f'Pages: {primary_page_df.page.nunique()}',
    f'Source groups: {primary_page_df.source_group.nunique()}',
    f'Eligible pixels: {len(primary)}',
    f'Primary CLEAN threshold: {PRIMARY_CLEAN_THRESHOLD}',
    f'Pixels selected to clean: {int(primary_summary["selected_clean_pixels"])}',
    f'Annotation pixels cleaned: {int(primary_summary["annotation_cleaned"])}',
    f'Genuine-print pixels lost: {int(primary_summary["print_lost"])}',
    f'Truth-uncertain pixels changed: {int(primary_summary["truth_uncertain_changed"])}',
    f'Utility: {primary_summary["utility"]}',
    f'Catastrophic components: {int(primary_summary["catastrophic_components"])}',
    f'CLEAN coverage: {primary_summary["clean_coverage"]:.6f}',
    f'UNCERTAIN rate: {primary_summary["uncertain_rate"]:.6f}','',
    'DEVELOPMENT FLAGS',
] + [f'- {k}: {v}' for k,v in gates.items()] + [
    '',
    'Interpretation: development OOF evidence only. No automatic promotion is authorised.',
    'A5 defaults to exact A1 wherever CLEAN is not validly emitted.',
]
(RUN/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines))

# 13. Final upload package

In [ ]:
# @title
UPLOAD_ARTEFACTS = [
    RUN/'executive_summary.txt',
    RUN/'notebook88_verdict.json',
    RUN/'experiment_identity.json',
    RUN/'software_environment.json',
    INPUTS/'channel_schema.json',
    INPUTS/'fold_map.csv',
    INPUTS/'page_role_manifest.csv',
    TABLES/'a5_threshold_sweep.csv',
    TABLES/'a5_primary_results_by_page.csv',
    TABLES/'a5_primary_results_by_source_group.csv',
    TABLES/'a5_visual_review_queue.csv',
    AUDIT/'source_group_split_audit.csv',
    AUDIT/'input_and_label_audit.csv',
    AUDIT/'model_inventory.csv',
    AUDIT/'calibrator_inventory.csv',
    VISUALS/'a5_safety_coverage_frontier.png',
    VISUALS/'most_print_loss.png',
    VISUALS/'most_annotation_cleanup.png',
    VISUALS/'most_changed.png',
]

for source in UPLOAD_ARTEFACTS:
    require(source.exists(), f'Missing upload artefact: {source}')
    shutil.copy2(source,UPLOAD/source.name)

manifest_rows=[]
for path in sorted(UPLOAD.iterdir()):
    if path.is_file():
        manifest_rows.append({
            'file':path.name,'bytes':path.stat().st_size,'sha256':sha256_file(path)
        })
pd.DataFrame(manifest_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)

readme = [
    'NOTEBOOK 88 A5 CONTEXTUAL SAFE-CLEANUP REVIEW PACKAGE','',
    'Start with executive_summary.txt and notebook88_verdict.json.',
    'Inspect a5_threshold_sweep.csv, page/source-group tables and the three contact sheets.',
    'Models and calibrators remain in the full run directory and are identified by experiment_identity.json.',
    'This package is development evidence only.',
]
(UPLOAD/'README.txt').write_text('\n'.join(readme))

zip_base=RUN/'notebook88_a5_contextual_safe_cleanup_bundle'
shutil.make_archive(str(zip_base),'zip',UPLOAD)
print('Bundle:',str(zip_base)+'.zip')
print(json.dumps(verdict,indent=2,default=str))